# Rastreabilidade Assistencial no SUS via Data Linkage
### Auditoria da População de Pacientes (jul de 2024 à jun de 2025) entre RNDS e SIA/SIH utilizando CPF/CNS como Identificador Único”

Avaliação da integridade e completude do fluxo de dados assistenciais no SUS, verificando a sobreposição e as exclusões (pacientes) entre os sistemas de Regulação (RNDS) e Faturamento (SIA/SIH) para o ano de 2024. A análise utiliza o CPF ou CNS como chave de linkage, alinhando-se aos princípios da Portaria 6.656/2025.

### Objetivo Principal:
Demonstrar a porcentagem de sobreposição de pacientes e procedimentos entre as bases. Especificamente:
•	Comprovar a rastreabilidade: Determinar a proporção de pacientes concluídos na RNDS (Regulação) que efetivamente aparecem nas bases de faturamento (SIA/SIH).
•	Identificar gargalos/inconsistências: Determinar a proporção de pacientes faturados (SIA/SIH) que não passaram ou não tiveram registro de conclusão na RNDS, evidenciando falhas no registro de Regulação Assistencial.
•	Validar a RNDS: Comprovar estatisticamente que os dados da RNDS, apesar de iniciais, já fornecem uma base válida para estudos de fluxo assistencial e tempo de espera.


### Metodologia de Ciência de Dados e Estatística (Foco em Linkage):
#### ETAPA 1: Analise e tratamento.
Devido ao volume de dados os arquivos estão em Parquet, veja o 'convert_csv_parquet.ipynb' 


In [1]:
# BIBLIOTECA
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pyarrow as pa
import pandas as pd
import os
from collections import Counter
from datetime import datetime

# Contagem de TEMPO de processamento
inicio = datetime.now()
print(f"🔵 Início da execução: {inicio.strftime('%H:%M:%S')}")

🔵 Início da execução: 18:34:35


In [2]:
# Função que resume informações de identificação em um arquivo Parquet
def resumo_identificacao_parquet(parquet_path, batch_size=500_000):
    # Abre o arquivo Parquet para leitura em lotes (batches)
    parquet_file = pq.ParquetFile(parquet_path)

    # Inicializa contadores
    total = sem_cpf = sem_cns = sem_ambos = 0

    # Itera sobre o arquivo em lotes de até 'batch_size' linhas
    # Lendo apenas as colunas "CPF_PAC" e "CNS_PAC"
    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=["CPF_PAC", "CNS_PAC"]
    ):
        # Cria máscaras booleanas indicando onde os valores estão vazios ("")
        cpf_vazio = pc.equal(batch["CPF_PAC"], "")
        cns_vazio = pc.equal(batch["CNS_PAC"], "")

        # Atualiza o total de linhas processadas
        total += batch.num_rows

        # Conta quantos registros têm CPF vazio
        sem_cpf += pc.sum(cpf_vazio).as_py()

        # Conta quantos registros têm CNS vazio
        sem_cns += pc.sum(cns_vazio).as_py()

        # Conta quantos registros têm CPF e CNS vazios ao mesmo tempo
        sem_ambos += pc.sum(pc.and_(cpf_vazio, cns_vazio)).as_py()

    # Retorna um dicionário com o resumo das contagens
    return {
        "GERAL": total,                  # Total de registros processados
        "SEM CPF": sem_cpf,              # Registros sem CPF
        "SEM CNS": sem_cns,              # Registros sem CNS
        "SEM CPF e SEM CNS": sem_ambos   # Registros sem ambos
    }


In [3]:
# Imprime um cabeçalho para indicar qual base está sendo analisada
print("===== BASE SIH =====")

# Chama a função resumo_identificacao_parquet para gerar o resumo
# do arquivo 'SIH.parquet' localizado na pasta 'base'
resumo_sih = resumo_identificacao_parquet(r"base/SIH.parquet")

# Percorre o dicionário retornado pela função (chaves e valores)
for k, v in resumo_sih.items():
    # Imprime cada chave (descrição) alinhada à esquerda em 25 caracteres
    # e o valor formatado com separador de milhar (ex.: 1,000,000)
    print(f"{k:<25} {v:,}")


===== BASE SIH =====
GERAL                     4,255,843
SEM CPF                   2,144,083
SEM CNS                   2,115,083
SEM CPF e SEM CNS         3,323


In [4]:
# Imprime um cabeçalho para indicar qual base está sendo analisada
print("===== BASE SIA =====")

# Chama a função resumo_identificacao_parquet para gerar o resumo
# do arquivo 'SIA.parquet' localizado na pasta 'base'
resumo_sia = resumo_identificacao_parquet(r"base/SIA.parquet")

# Percorre o dicionário retornado pela função (chaves e valores)
for k, v in resumo_sia.items():
    # Imprime cada chave (descrição) alinhada à esquerda em 25 caracteres
    # e o valor formatado com separador de milhar (ex.: 1,000,000 → 1.000.000)
    print(f"{k:<25} {v:,}")


===== BASE SIA =====
GERAL                     176,260,340
SEM CPF                   171,133,678
SEM CNS                   36,485,231
SEM CPF e SEM CNS         35,622,906


In [5]:
# Função para limpar um arquivo Parquet, removendo registros sem CPF e sem CNS
def limpar_parquet_sem_cpf_e_cns(
    parquet_in: str,          # caminho do arquivo de entrada (.parquet)
    parquet_out: str,         # caminho do arquivo de saída (.parquet)
    compression: str = "zstd",# tipo de compressão usada ao salvar
    batch_size: int = 500_000 # tamanho dos lotes de leitura (para não estourar memória)
):
    # Abre o arquivo Parquet para leitura em lotes
    parquet_file = pq.ParquetFile(parquet_in)
    writer = None  # será usado para escrever o novo arquivo

    try:
        # Itera sobre o arquivo em lotes de até 'batch_size' linhas
        for batch in parquet_file.iter_batches(batch_size=batch_size):
            # Converte o lote em uma tabela PyArrow
            table = pa.Table.from_batches([batch])

            # Cria condição booleana: CPF vazio E CNS vazio
            cond_remover = pc.and_(
                pc.equal(table["CPF_PAC"], ""),
                pc.equal(table["CNS_PAC"], "")
            )

            # Filtra a tabela invertendo a condição (mantém apenas registros válidos)
            table_limpa = pc.filter(table, pc.invert(cond_remover))

            # Se o lote filtrado não tiver linhas, pula para o próximo
            if table_limpa.num_rows == 0:
                continue

            # Inicializa o writer na primeira vez que encontra dados válidos
            if writer is None:
                writer = pq.ParquetWriter(
                    parquet_out,
                    schema=table.schema,   # usa o schema do lote
                    compression=compression,
                    use_dictionary=True,   # otimiza armazenamento
                )

            # Escreve o lote filtrado no arquivo de saída
            writer.write_table(table_limpa)

    finally:
        # Fecha o writer ao final (garante que o arquivo seja salvo corretamente)
        if writer:
            writer.close()


In [6]:
# Mensagem inicial para indicar que a limpeza da base SIH começou
print("🧹 Limpando SIH...")

# Chama a função de limpeza, removendo registros sem CPF e sem CNS
# - parquet_in: arquivo original
# - parquet_out: arquivo limpo que será criado
limpar_parquet_sem_cpf_e_cns(
    parquet_in=r"base/SIH.parquet",
    parquet_out=r"base/SIH_LIMPO.parquet",
)

# Mensagem final confirmando que o arquivo limpo foi gerado
print("✅ SIH limpo criado")


🧹 Limpando SIH...
✅ SIH limpo criado


In [7]:
# Mensagem inicial para indicar que a limpeza da base SIA começou
print("🧹 Limpando SIA...")

# Chama a função de limpeza, removendo registros sem CPF e sem CNS
limpar_parquet_sem_cpf_e_cns(
    parquet_in=r"base/SIA.parquet",
    parquet_out=r"base/SIA_LIMPO.parquet",
)

# Mensagem final confirmando que o arquivo limpo foi gerado
print("✅ SIA limpo criado")


🧹 Limpando SIA...
✅ SIA limpo criado


## BASE DA REGULAÇÃO

In [8]:
# Caminhos dos arquivos de entrada e saída
arquivo_in = r"base/RNDS.parquet"              # Arquivo original
arquivo_out = r"base/RNDS_renomeado.parquet"   # Arquivo que será gerado com colunas renomeadas

# Dicionário de renomeação: mapeia nomes antigos → novos
mapa_colunas = {
    "nu_cpf_paciente": "CPF_PAC",
    "nu_cns_paciente": "CNS_PAC",
    "co_sigtap": "COD_SIGTAP_PROCEDIMENTO",
    "co_cbo": "CBO",
    "sg_uf_estab_executante": "UF_DESC_ATEND",
    "co_municipio_estab_executante": "IBGE_ATEND",
    "co_cnes_estab_executante": "CNES_ATEND",
    "data_solicitacao": "DATA_SOLICITACAO",
    "data_autorizacao": "DATA_AUTORIZACAO",
    "data_execucao": "DATA_EXECUCAO",
    "st_vida_paciente": "ST_VIDA",
    "st_solicitacao": "STATUS",
    "id_registro_sistema_origem": "ID_ORIGEM",
    "ds_sistema_origem": "SIST_ORIGEM"
}

# Abre o arquivo Parquet original para leitura em lotes
parquet_file = pq.ParquetFile(arquivo_in)

# Inicializa o writer (será criado apenas quando necessário)
writer = None

# Processa o arquivo em lotes de até 500 mil linhas
for batch in parquet_file.iter_batches(batch_size=500_000):
    # Converte o lote em uma tabela PyArrow
    table = pa.Table.from_batches([batch])

    # Renomeia as colunas usando o dicionário de mapeamento
    # Se a coluna não estiver no dicionário, mantém o nome original
    table = table.rename_columns([mapa_colunas.get(c, c) for c in table.schema.names])

    # Cria o writer na primeira vez que encontra dados
    if writer is None:
        writer = pq.ParquetWriter(arquivo_out, table.schema)

    # Escreve o lote já com colunas renomeadas no novo arquivo
    writer.write_table(table)

# Fecha o writer ao final para garantir que o arquivo seja salvo corretamente
if writer:
    writer.close()

# Mensagem final confirmando a criação do arquivo renomeado
print("✅ Arquivo renomeado salvo em:", arquivo_out)


✅ Arquivo renomeado salvo em: base/RNDS_renomeado.parquet


In [9]:
# Imprime um cabeçalho para indicar qual base está sendo analisada
print("===== BASE RNDS =====")

# Chama a função resumo_identificacao_parquet para gerar o resumo
# do arquivo 'RNDS_renomeado.parquet' localizado na pasta 'base'
resumo_rnds = resumo_identificacao_parquet(r"base/RNDS_renomeado.parquet")

# Percorre o dicionário retornado pela função (chaves e valores)
for k, v in resumo_rnds.items():
    # Imprime cada chave (descrição) alinhada à esquerda em 25 caracteres
    # e o valor formatado com separador de milhar (ex.: 1,000,000 → 1.000.000)
    print(f"{k:<25} {v:,}")


===== BASE RNDS =====
GERAL                     107,600,060
SEM CPF                   1,541,035
SEM CNS                   0
SEM CPF e SEM CNS         0


In [10]:
# Função para limpar um arquivo Parquet, removendo registros sem CPF
def limpar_parquet_sem_cpf(
    parquet_in: str,          # caminho do arquivo de entrada (.parquet)
    parquet_out: str,         # caminho do arquivo de saída (.parquet)
    compression: str = "zstd",# tipo de compressão usada ao salvar
    batch_size: int = 500_000 # tamanho dos lotes de leitura (para não estourar memória)
):
    # Abre o arquivo Parquet para leitura em lotes
    parquet_file = pq.ParquetFile(parquet_in)
    writer = None  # será usado para escrever o novo arquivo

    try:
        # Itera sobre o arquivo em lotes de até 'batch_size' linhas
        for batch in parquet_file.iter_batches(batch_size=batch_size):
            # Converte o lote em uma tabela PyArrow
            table = pa.Table.from_batches([batch])

            # Cria condição booleana: CPF vazio
            cond_remover = pc.equal(table["CPF_PAC"], "")

            # Filtra a tabela invertendo a condição (mantém apenas registros com CPF preenchido)
            table_limpa = pc.filter(table, pc.invert(cond_remover))

            # Se o lote filtrado não tiver linhas, pula para o próximo
            if table_limpa.num_rows == 0:
                continue

            # Inicializa o writer na primeira vez que encontra dados válidos
            if writer is None:
                writer = pq.ParquetWriter(
                    parquet_out,
                    schema=table.schema,   # usa o schema do lote
                    compression=compression,
                    use_dictionary=True,   # otimiza armazenamento
                )

            # Escreve o lote filtrado no arquivo de saída
            writer.write_table(table_limpa)

    finally:
        # Fecha o writer ao final (garante que o arquivo seja salvo corretamente)
        if writer:
            writer.close()


In [11]:
# Mensagem inicial para indicar que a limpeza da base RNDS começou
print("🧹 Limpando RNDS...")

# Chama a função de limpeza, removendo registros sem CPF
# - parquet_in: arquivo original (já renomeado anteriormente)
# - parquet_out: arquivo limpo que será criado contendo apenas registros com CPF
limpar_parquet_sem_cpf(
    parquet_in=r"base/RNDS_renomeado.parquet",
    parquet_out=r"base/RNDS_com_cpf.parquet",
)

# Mensagem final confirmando que o arquivo limpo foi gerado
print("✅ RNDS limpo criado")


🧹 Limpando RNDS...
✅ RNDS limpo criado


In [12]:
# Nome da coluna que indica status de vida
COL_STATUS = "ST_VIDA"   # ← AjustE: 1 = VIVO, 0 = MORTO

# Função para contar quantos pacientes estão vivos ou mortos em um arquivo Parquet
def contar_vida_parquet(parquet_path, batch_size=500_000):
    # Abre o arquivo Parquet para leitura em lotes
    parquet_file = pq.ParquetFile(parquet_path)
    
    # Inicializa um contador (do módulo collections.Counter)
    contador = Counter()

    # Itera sobre o arquivo em lotes de até 'batch_size' linhas
    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=[COL_STATUS]   # lê apenas a coluna ST_VIDA
    ):
        # Extrai a coluna ST_VIDA do lote
        arr = batch[COL_STATUS]

        # Remove valores nulos (caso existam)
        arr = pc.drop_null(arr)

        # Converte para lista Python (ex.: [1, 0, 1, 1, 0])
        valores = arr.to_pylist()

        # Atualiza o contador com os valores encontrados
        contador.update(valores)

    # Retorna o contador com a frequência de cada valor
    return contador


In [13]:
# Conta os registros de vida/morte no arquivo RNDS_com_cpf.parquet
contagem = contar_vida_parquet(r"base/RNDS_com_cpf.parquet")

# Obtém os totais com valor padrão 0 caso não exista
total_vivos = contagem.get(1, 0)
total_mortos = contagem.get(0, 0)

# Impressão no formato solicitado
print("===== PACIENTES NA LISTA =====")
print(f"VIVO   {total_vivos:,}")
print(f"MORTO  {total_mortos:,}")


===== PACIENTES NA LISTA =====
VIVO   105,523,722
MORTO  535,303


In [14]:
# Nome da coluna que indica o status da solicitação
COL_STATUS = "STATUS"   # ← Ajuste: define qual coluna será usada

# Função para contar a frequência dos diferentes status em um arquivo Parquet
def contar_status_parquet(parquet_path, batch_size=500_000):
    # Abre o arquivo Parquet para leitura em lotes
    parquet_file = pq.ParquetFile(parquet_path)
    
    # Inicializa um contador (do módulo collections.Counter)
    contador = Counter()

    # Itera sobre o arquivo em lotes de até 'batch_size' linhas
    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=[COL_STATUS]   # lê apenas a coluna STATUS
    ):
        # Extrai a coluna STATUS do lote
        arr = batch[COL_STATUS]

        # Remove valores nulos (caso existam)
        arr = pc.drop_null(arr)

        # Converte para lista Python (ex.: ["AUTORIZADO", "NEGADO", "PENDENTE"])
        valores = arr.to_pylist()

        # Atualiza o contador com os valores encontrados
        contador.update(valores)

    # Retorna o contador com a frequência de cada status
    return contador


In [15]:
# Conta os registros de status no arquivo RNDS_com_cpf.parquet
contagem = contar_status_parquet(r"base/RNDS_com_cpf.parquet")

print("===== STATUS RNDS =====")

# Percorre os resultados do contador, ordenados do mais frequente para o menos
for status, total in contagem.most_common():
    print(f"{status:<30} {total:,}")


===== STATUS RNDS =====
Atendido/Internado             64,707,871
Agendado                       28,592,746
Falta                          6,180,287
Pendente                       5,859,409
Negado/Cancelado               466,298
Devolvido para o solicitante.  130,649
Excluído                       121,765


In [16]:
# Função para filtrar registros de um arquivo Parquet,
# removendo aqueles que possuem determinados valores na coluna STATUS
def filtrar_status_parquet(
    parquet_in,        # caminho do arquivo de entrada (.parquet)
    parquet_out,       # caminho do arquivo de saída (.parquet)
    status_excluir,    # lista de valores de STATUS que devem ser removidos
    batch_size=500_000 # tamanho dos lotes de leitura (para não estourar memória)
):
    # Abre o arquivo Parquet para leitura em lotes
    parquet_file = pq.ParquetFile(parquet_in)
    writer = None  # será usado para escrever o novo arquivo

    try:
        # Itera sobre o arquivo em lotes de até 'batch_size' linhas
        for batch in parquet_file.iter_batches(batch_size=batch_size):
            # Converte o lote em uma tabela PyArrow
            table = pa.Table.from_batches([batch])

            # Cria condição booleana: STATUS está dentro da lista de exclusão
            cond_excluir = pc.is_in(
                table[COL_STATUS],
                value_set=pa.array(status_excluir)
            )

            # Inverte a condição → mantém apenas registros cujo STATUS não está na lista
            table_ok = pc.filter(table, pc.invert(cond_excluir))

            # Se não houver registros válidos nesse lote, pula para o próximo
            if table_ok.num_rows == 0:
                continue

            # Inicializa o writer na primeira vez que encontra dados válidos
            if writer is None:
                writer = pq.ParquetWriter(
                    parquet_out,
                    schema=table_ok.schema,
                    compression="zstd"   # compressão padrão
                )

            # Escreve o lote filtrado no arquivo de saída
            writer.write_table(table_ok)

    finally:
        # Fecha o writer ao final (garante que o arquivo seja salvo corretamente)
        if writer:
            writer.close()


In [17]:
# Chama a função para filtrar registros do arquivo RNDS_renomeado.parquet
# Exclui todos os registros cujo STATUS esteja em uma das opções da lista
filtrar_status_parquet(
    parquet_in=r"base/RNDS_renomeado.parquet",          # arquivo de entrada
    parquet_out=r"base/RNDS_status_agendados.parquet",  # arquivo de saída filtrado
    status_excluir=[                                    # lista de status a excluir
        "Falta",
        "Pendente",
        "Negado/Cancelado",
        "Devolvido para o solicitante.",
        "Excluído"
    ]
)

# Mensagem final confirmando que o arquivo filtrado foi criado
print("✅ RNDS filtrada por status criada")


✅ RNDS filtrada por status criada


In [18]:
# Excluir arquivos que não serão utilizados... 

# Lista de arquivos que devem ser removidos
lista_arquivos = [
    r"base/RNDS_vivos.parquet",
    r"base/RNDS_renomeado.parquet",
    r"base/RNDS_com_cpf.parquet"
]

# Percorre cada arquivo da lista
for arquivo in lista_arquivos:
    # Verifica se o arquivo existe
    if os.path.exists(arquivo):
        # Remove o arquivo
        os.remove(arquivo)
        print(f"{arquivo} removido.")
    else:
        # Caso não exista, informa ao usuário
        print(f"{arquivo} não encontrado.")


base/RNDS_vivos.parquet não encontrado.
base/RNDS_renomeado.parquet removido.
base/RNDS_com_cpf.parquet removido.


In [19]:
# Função para extrair todos os valores únicos da coluna CNS_PAC em um arquivo Parquet
def extrair_cns_sia(parquet_sia, batch_size=500_000):
    # Abre o arquivo Parquet para leitura em lotes
    parquet_file = pq.ParquetFile(parquet_sia)
    
    # Cria um conjunto vazio (set) para armazenar os valores únicos de CNS
    cns_set = set()

    # Itera sobre o arquivo em lotes de até 'batch_size' linhas
    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=["CNS_PAC"]   # lê apenas a coluna CNS_PAC
    ):
        # Extrai a coluna CNS_PAC do lote
        col = batch["CNS_PAC"]

        # Remove valores nulos
        col = pc.drop_null(col)

        # Garante que os valores sejam únicos dentro do lote
        col = pc.unique(col)

        # Converte para lista Python e adiciona ao conjunto global
        cns_set.update(col.to_pylist())

    # Retorna o conjunto com todos os CNS únicos encontrados
    return cns_set


In [20]:
# Extrai todos os CNS únicos da base SIH_LIMPO.parquet
cns_sih = extrair_cns_sia(r"base/SIH_LIMPO.parquet")

# Imprime a quantidade total de valores distintos encontrados
print(f"CNS distintos no SIH: {len(cns_sih):,}")

CNS distintos no SIH: 1,876,043


In [21]:
# Extrai todos os CNS únicos da base SIA_LIMPO.parquet
cns_sia = extrair_cns_sia(r"base/SIA_LIMPO.parquet")

# Imprime a quantidade total de valores distintos encontrados
# Usa separador de milhar para facilitar a leitura
print(f"CNS distintos no SIA: {len(cns_sia):,}")


CNS distintos no SIA: 28,218,255


In [22]:
# Célula [dc65a3b8]
# Combina os CNS únicos do SIA e do SIH
cns_combinado = cns_sia.union(cns_sih)

# Calcula a sobreposição (interseção)
cns_sobreposicao = cns_sia.intersection(cns_sih)

# Calcula quantos CNS são exclusivos do SIH (enriquecendo o conjunto do SIA)
cns_exclusivos_sih = cns_sih.difference(cns_sia)

print("===== ANÁLISE DE UNICIDADE E ENRIQUECIMENTO (CNS) =====")
print(f"CNS distintos no SIA: {len(cns_sia):,}")
print(f"CNS distintos no SIH: {len(cns_sih):,}")
print(f"Sobreposição (CNS presentes em SIA e SIH): {len(cns_sobreposicao):,}")
print(f"CNS Exclusivos no SIH (Novos chaves adicionadas): {len(cns_exclusivos_sih):,}")
print(f"Total de CNS Únicos Combinados (SIA + SIH): {len(cns_combinado):,}")
print("=======================================================")

===== ANÁLISE DE UNICIDADE E ENRIQUECIMENTO (CNS) =====
CNS distintos no SIA: 28,218,255
CNS distintos no SIH: 1,876,043
Sobreposição (CNS presentes em SIA e SIH): 866,839
CNS Exclusivos no SIH (Novos chaves adicionadas): 1,009,204
Total de CNS Únicos Combinados (SIA + SIH): 29,227,459


In [23]:
# CÓDIGO FASE 1: CRIAÇÃO DO MAPA ÚNICO CNS -> CPF A PARTIR DA RNDS

import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
import os

# Arquivo de Origem (RNDS) e Arquivo de Destino (Mapa)
RNDS_FILE = r"base/RNDS_status_agendados.parquet"
MAP_OUTPUT_FILE = r"base/RNDS_CNS_CPF_MAP.parquet"

BATCH_SIZE = 5_000_000
cns_to_cpf_map = {}

# --- Função de Extração de Dados Otimizada ---
def create_rnds_map_optimized(file_path):
    """Extrai pares CNS_PAC e CPF_PAC da RNDS em blocos e atualiza o mapa global."""
    global cns_to_cpf_map
    
    print(f"Extraindo pares CNS/CPF de: {file_path} (Fonte RNDS)")
    
    if not os.path.exists(file_path):
        print(f"ERRO: Arquivo não encontrado: {file_path}. Verifique o caminho.")
        return

    parquet_file = pq.ParquetFile(file_path)
    
    # Itera em blocos lendo apenas as colunas essenciais
    for i, batch in enumerate(parquet_file.iter_batches(columns=["CNS_PAC", "CPF_PAC"], batch_size=BATCH_SIZE)):
        batch_df = batch.to_pandas()
        
        # Filtra registros onde o CNS ou CPF não é nulo
        # Somente registros com ambas as chaves são úteis para o mapeamento
        clean_df = batch_df.dropna(subset=['CNS_PAC', 'CPF_PAC'])
        
        # Cria um novo mapa local a partir do bloco (CNS como chave, CPF como valor)
        new_mappings = clean_df.set_index('CNS_PAC')['CPF_PAC'].to_dict()
        
        # Atualiza o mapa global. A chave CNS é única, substituindo o último CPF encontrado (o que garante a unicidade)
        cns_to_cpf_map.update(new_mappings)
        
        if (i + 1) % 5 == 0:
            print(f"  {i+1} blocos processados. Mapeamentos únicos no dicionário: {len(cns_to_cpf_map):,}")

# 1. Coleta os dados e constrói o mapa único a partir da RNDS
create_rnds_map_optimized(RNDS_FILE)

print(f"\nTotal de Mapeamentos CNS -> CPF Únicos criados a partir da RNDS: {len(cns_to_cpf_map):,}")

# 2. Converte o dicionário final para DataFrame
unique_map_df = pd.DataFrame(list(cns_to_cpf_map.items()), columns=['CNS_PAC', 'CPF_PAC'])

# 3. Salva o Mapa de Ligação no disco
# Usando pq.write_table para evitar o AttributeError
pq.write_table(pa.Table.from_pandas(unique_map_df), MAP_OUTPUT_FILE)

print(f"✅ Mapa de Mapeamento Único CNS -> CPF (RNDS) criado: {MAP_OUTPUT_FILE}")

# Limpar memória
del cns_to_cpf_map
del unique_map_df

Extraindo pares CNS/CPF de: base/RNDS_status_agendados.parquet (Fonte RNDS)
  5 blocos processados. Mapeamentos únicos no dicionário: 8,577,651
  10 blocos processados. Mapeamentos únicos no dicionário: 11,694,595
  15 blocos processados. Mapeamentos únicos no dicionário: 13,856,124

Total de Mapeamentos CNS -> CPF Únicos criados a partir da RNDS: 15,118,510
✅ Mapa de Mapeamento Único CNS -> CPF (RNDS) criado: base/RNDS_CNS_CPF_MAP.parquet


In [24]:
# Imprime um cabeçalho para indicar qual base está sendo analisada
print("===== BASE MAPA (CPF E CNS UNICOS NA RNDS) =====")

# Chama a função resumo_identificacao_parquet para gerar o resumo
# do arquivo 'SIA.parquet' localizado na pasta 'base'
resumo_sia = resumo_identificacao_parquet(r"base/RNDS_CNS_CPF_MAP.parquet")

# Percorre o dicionário retornado pela função (chaves e valores)
for k, v in resumo_sia.items():
    # Imprime cada chave (descrição) alinhada à esquerda em 25 caracteres
    # e o valor formatado com separador de milhar (ex.: 1,000,000 → 1.000.000)
    print(f"{k:<30} {v:,}")

===== BASE MAPA (CPF E CNS UNICOS NA RNDS) =====
GERAL                          15,118,510
SEM CPF                        286,401
SEM CNS                        0
SEM CPF e SEM CNS              0


In [25]:
# Imprime um cabeçalho para indicar qual base está sendo analisada
print("===== BASE RNDS =====")

# Chama a função resumo_identificacao_parquet para gerar o resumo
# do arquivo 'SIA.parquet' localizado na pasta 'base'
resumo_sia = resumo_identificacao_parquet(r"base/RNDS_status_agendados.parquet")

# Percorre o dicionário retornado pela função (chaves e valores)
for k, v in resumo_sia.items():
    # Imprime cada chave (descrição) alinhada à esquerda em 25 caracteres
    # e o valor formatado com separador de milhar (ex.: 1,000,000 → 1.000.000)
    print(f"{k:<30} {v:,}")

===== BASE RNDS =====
GERAL                          94,606,093
SEM CPF                        1,305,476
SEM CNS                        0
SEM CPF e SEM CNS              0


In [26]:
# Função para filtrar registros da RNDS com base nos CNS válidos do SIA
def extrair_rnds_filtrada(
    parquet_rnds,     # arquivo RNDS de entrada (.parquet)
    cns_validos,      # conjunto/lista de CNS válidos (extraídos do SIA)
    parquet_out,      # arquivo de saída (.parquet) com registros filtrados
    batch_size=500_000 # tamanho dos lotes de leitura (para não estourar memória)
):
    # Abre o arquivo RNDS para leitura em lotes
    parquet_file = pq.ParquetFile(parquet_rnds)
    writer = None  # será usado para escrever o novo arquivo

    try:
        # Itera sobre o arquivo em lotes de até 'batch_size' linhas
        for batch in parquet_file.iter_batches(
            batch_size=batch_size,
            # CORREÇÃO: usar os nomes de coluna padronizados
            columns=["CNS_PAC", "CPF_PAC"] 
        ):
            # Converte o lote em uma tabela PyArrow
            table = pa.Table.from_batches([batch])

            # Cria condição: CNS_PAC está dentro da lista de CNS válidos
            cond = pc.is_in(
                table["CNS_PAC"], 
                value_set=pa.array(list(cns_validos))
            )

            # Aplica o filtro → mantém apenas registros com CNS válido
            table_filtrada = pc.filter(table, cond)

            # Se não houver registros nesse lote, pula para o próximo
            if table_filtrada.num_rows == 0:
                continue

            # Inicializa o writer na primeira vez que encontra dados válidos
            if writer is None:
                writer = pq.ParquetWriter(
                    parquet_out,
                    schema=table_filtrada.schema,
                    compression="zstd"   # compressão padrão
                )

            # Escreve o lote filtrado no arquivo de saída
            writer.write_table(table_filtrada)

    finally:
        # Fecha o writer ao final (garante que o arquivo seja salvo corretamente)
        if writer:
            writer.close()


In [27]:
# 3. Filtra a RNDS usando o conjunto unificado de CNS
extrair_rnds_filtrada(
    parquet_rnds=r"base/RNDS_status_agendados.parquet", # RNDS já limpa e filtrada por status
    cns_validos=cns_combinado,                         # Conjunto UNIFICADO de CNS (SIA + SIH)
    parquet_out=r"base/RNDS_filtrada_pelo_Faturamento_CNS.parquet" # Novo arquivo RNDS filtrado
)
print("✅ RNDS filtrada pelo CNS do Faturamento (SIA/SIH) criada")

✅ RNDS filtrada pelo CNS do Faturamento (SIA/SIH) criada


In [28]:
# Extrair CNS únicos da RNDS filtrada (status agendados/atendidos)
cns_rnds = extrair_cns_sia(r"base/RNDS_status_agendados.parquet")
print(f"CNS distintos na RNDS (status válidos): {len(cns_rnds):,}")

# Já temos cns_sia e cns_sih das células anteriores
# Calcular sobreposição
sobreposicao_rnds_sia = cns_rnds.intersection(cns_sia)
sobreposicao_rnds_sih = cns_rnds.intersection(cns_sih)
sobreposicao_rnds_faturamento = cns_rnds.intersection(cns_combinado)  # União SIA + SIH

# Porcentagens
total_rnds = len(cns_rnds)
perc_sia = len(sobreposicao_rnds_sia) / total_rnds * 100
perc_sih = len(sobreposicao_rnds_sih) / total_rnds * 100
perc_faturamento = len(sobreposicao_rnds_faturamento) / total_rnds * 100

print("\n===== LINKAGE RNDS vs FATURAMENTO (CNS) =====")
print(f"Total de pacientes na RNDS (status válidos): {total_rnds:,}")
print(f"\nSobreposição RNDS vs SIA: {len(sobreposicao_rnds_sia):,} ({perc_sia:.2f}%)")
print(f"Sobreposição RNDS vs SIH: {len(sobreposicao_rnds_sih):,} ({perc_sih:.2f}%)")
print(f"Sobreposição RNDS vs SIA+SIH: {len(sobreposicao_rnds_faturamento):,} ({perc_faturamento:.2f}%)")

CNS distintos na RNDS (status válidos): 15,118,510

===== LINKAGE RNDS vs FATURAMENTO (CNS) =====
Total de pacientes na RNDS (status válidos): 15,118,510

Sobreposição RNDS vs SIA: 6,371,235 (42.14%)
Sobreposição RNDS vs SIH: 460,155 (3.04%)
Sobreposição RNDS vs SIA+SIH: 6,552,718 (43.34%)


In [29]:
# Pacientes faturados (SIA+SIH) que passaram pela regulação (RNDS)
total_faturados = len(cns_combinado)
perc_faturados_com_regulacao = len(sobreposicao_rnds_faturamento) / total_faturados * 100

print(f"\n===== PERSPECTIVA INVERSA =====")
print(f"Total de pacientes faturados (SIA+SIH): {total_faturados:,}")
print(f"Pacientes faturados que passaram pela RNDS: {len(sobreposicao_rnds_faturamento):,} ({perc_faturados_com_regulacao:.2f}%)")


===== PERSPECTIVA INVERSA =====
Total de pacientes faturados (SIA+SIH): 29,227,459
Pacientes faturados que passaram pela RNDS: 6,552,718 (22.42%)


In [30]:
# ==================== ANÁLISE DISK-BASED - USA DISCO, NÃO MEMÓRIA ====================
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
from datetime import datetime
import os
import sqlite3

# ==================== CONFIGURAÇÕES ====================
BATCH_SIZE = 100_000
OUTPUT_DIR = "analise_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("🔵 ANÁLISE DE RASTREABILIDADE - DISK-BASED")
print("="*80)
print("Período: Jul/2024 a Jun/2025")
print("Modo: Usa disco (SQLite) em vez de memória RAM")
print("="*80)

# ==================== FUNÇÕES ====================

def extrair_mes_ano_rnds(data_str):
    if not data_str or str(data_str).strip() in ['', 'None', 'nan']:
        return None
    data_str = str(data_str).strip().split(' ')[0]
    try:
        if '/' in data_str:
            partes = data_str.split('/')
            if len(partes) == 3:
                return f"{partes[1].zfill(2)}{partes[2]}"
        elif '-' in data_str:
            partes = data_str.split('-')
            if len(partes) == 3:
                return f"{partes[1].zfill(2)}{partes[0]}"
    except:
        pass
    return None

def extrair_mes_ano_faturamento(data_str):
    if not data_str or str(data_str).strip() in ['', 'None', 'nan']:
        return None
    data_str = str(data_str).strip()
    try:
        if '/' in data_str:
            partes = data_str.split('/')
            if len(partes) == 2:
                return f"{partes[0].zfill(2)}{partes[1]}"
    except:
        pass
    return None

def criar_chave(cpf, cns, sigtap, mes_ano):
    identificador = cpf if cpf and cpf != '' else cns
    if identificador and sigtap and mes_ano:
        identificador = ''.join(filter(str.isdigit, str(identificador)))
        sigtap = ''.join(filter(str.isdigit, str(sigtap)))
        if len(identificador) in [11, 15] and len(sigtap) == 10:
            return f"{identificador}|{sigtap}|{mes_ano}"
    return None

def extrair_chaves_direto(parquet_in, parquet_out, extrair_data_fn, col_data):
    """Extrai chaves e salva DIRETAMENTE"""
    print(f"\n📊 {os.path.basename(parquet_in)}")
    
    if not os.path.exists(parquet_in):
        print(f"❌ Não encontrado")
        return 0
    
    total_proc = 0
    total_val = 0
    writer = None
    inicio = datetime.now()
    
    pq_file = pq.ParquetFile(parquet_in)
    
    try:
        for idx, batch in enumerate(pq_file.iter_batches(
            batch_size=BATCH_SIZE,
            columns=['CPF_PAC', 'CNS_PAC', 'COD_SIGTAP_PROCEDIMENTO', col_data]
        )):
            total_proc += batch.num_rows
            
            cpfs = batch['CPF_PAC'].to_pylist()
            cnss = batch['CNS_PAC'].to_pylist()
            sigtaps = batch['COD_SIGTAP_PROCEDIMENTO'].to_pylist()
            datas = batch[col_data].to_pylist()
            
            chaves = []
            for i in range(len(cpfs)):
                mes_ano = extrair_data_fn(datas[i])
                chave = criar_chave(cpfs[i], cnss[i], sigtaps[i], mes_ano)
                if chave:
                    chaves.append(chave)
                    total_val += 1
            
            if chaves:
                table = pa.table({'CHAVE': chaves})
                if writer is None:
                    writer = pq.ParquetWriter(parquet_out, table.schema, compression='zstd')
                writer.write_table(table)
                chaves.clear()
            
            if (idx + 1) % 50 == 0:
                print(f"  {total_proc:,} proc | {total_val:,} válidas")
    
    finally:
        if writer:
            writer.close()
    
    tempo = datetime.now() - inicio
    print(f"✅ {total_val:,} chaves | {tempo}")
    return total_val

# ==================== LINKAGE DISK-BASED COM SQLITE ====================

def linkage_sqlite(arquivo_rnds, arquivo_faturamento):
    """
    Usa SQLite para fazer o linkage (disco em vez de RAM)
    """
    print(f"\n🔗 LINKAGE DISK-BASED (SQLite)")
    
    db_path = f"{OUTPUT_DIR}/temp_linkage.db"
    
    # Remover DB anterior se existir
    if os.path.exists(db_path):
        os.remove(db_path)
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Criar tabelas
    cursor.execute('''
        CREATE TABLE rnds (
            chave TEXT PRIMARY KEY
        )
    ''')
    
    cursor.execute('''
        CREATE TABLE faturamento (
            chave TEXT
        )
    ''')
    
    cursor.execute('CREATE INDEX idx_fat ON faturamento(chave)')
    
    conn.commit()
    
    # 1. Carregar RNDS no SQLite
    print("  Fase 1: Carregando RNDS no SQLite...")
    total_rnds = 0
    
    pq_rnds = pq.ParquetFile(arquivo_rnds)
    for batch in pq_rnds.iter_batches(batch_size=BATCH_SIZE, columns=['CHAVE']):
        chaves = batch['CHAVE'].to_pylist()
        
        # Inserir em lote
        cursor.executemany('INSERT OR IGNORE INTO rnds VALUES (?)', [(c,) for c in chaves])
        total_rnds += len(chaves)
        
        if total_rnds % 10_000_000 == 0:
            conn.commit()
            print(f"    RNDS: {total_rnds:,}")
    
    conn.commit()
    print(f"  ✅ RNDS: {total_rnds:,} registros inseridos")
    
    # 2. Carregar Faturamento no SQLite
    print("\n  Fase 2: Carregando Faturamento no SQLite...")
    total_faturamento = 0
    
    pq_fat = pq.ParquetFile(arquivo_faturamento)
    for batch in pq_fat.iter_batches(batch_size=BATCH_SIZE, columns=['CHAVE']):
        chaves = batch['CHAVE'].to_pylist()
        
        cursor.executemany('INSERT INTO faturamento VALUES (?)', [(c,) for c in chaves])
        total_faturamento += len(chaves)
        
        if total_faturamento % 10_000_000 == 0:
            conn.commit()
            print(f"    Faturamento: {total_faturamento:,}")
    
    conn.commit()
    print(f"  ✅ Faturamento: {total_faturamento:,} registros inseridos")
    
    # 3. Calcular estatísticas com SQL
    print("\n  Fase 3: Calculando estatísticas (SQL)...")
    
    # Total de chaves únicas na RNDS
    cursor.execute('SELECT COUNT(*) FROM rnds')
    rnds_unicas = cursor.fetchone()[0]
    print(f"    RNDS únicas: {rnds_unicas:,}")
    
    # Matches (RNDS que aparecem no faturamento)
    print("    Calculando matches...")
    cursor.execute('''
        SELECT COUNT(DISTINCT f.chave)
        FROM faturamento f
        INNER JOIN rnds r ON f.chave = r.chave
    ''')
    matches = cursor.fetchone()[0]
    print(f"    Matches: {matches:,}")
    
    # Faturamento sem RNDS
    print("    Calculando gaps...")
    cursor.execute('''
        SELECT COUNT(*)
        FROM faturamento f
        WHERE NOT EXISTS (
            SELECT 1 FROM rnds r WHERE r.chave = f.chave
        )
    ''')
    gaps_faturamento = cursor.fetchone()[0]
    print(f"    Faturamento sem RNDS: {gaps_faturamento:,}")
    
    # RNDS não faturado
    rnds_nao_faturado = rnds_unicas - matches
    
    # Percentuais
    perc_match = (matches / total_rnds * 100) if total_rnds > 0 else 0
    perc_faturamento_sem_rnds = (gaps_faturamento / total_faturamento * 100) if total_faturamento > 0 else 0
    
    # 4. Exportar resultados para Parquet
    print("\n  Fase 4: Exportando resultados...")
    
    # Matches
    print("    Exportando matches...")
    cursor.execute('''
        SELECT DISTINCT f.chave
        FROM faturamento f
        INNER JOIN rnds r ON f.chave = r.chave
    ''')
    
    writer_match = None
    batch_matches = []
    for row in cursor:
        batch_matches.append(row[0])
        
        if len(batch_matches) >= BATCH_SIZE:
            table = pa.table({'CHAVE': batch_matches})
            if writer_match is None:
                writer_match = pq.ParquetWriter(
                    f"{OUTPUT_DIR}/matches_rnds_faturamento.parquet",
                    table.schema,
                    compression='zstd'
                )
            writer_match.write_table(table)
            batch_matches.clear()
    
    if batch_matches:
        table = pa.table({'CHAVE': batch_matches})
        if writer_match is None:
            writer_match = pq.ParquetWriter(
                f"{OUTPUT_DIR}/matches_rnds_faturamento.parquet",
                table.schema,
                compression='zstd'
            )
        writer_match.write_table(table)
    
    if writer_match:
        writer_match.close()
    
    print(f"    ✅ matches_rnds_faturamento.parquet")
    
    # Gaps faturamento
    print("    Exportando gaps faturamento...")
    cursor.execute('''
        SELECT f.chave
        FROM faturamento f
        WHERE NOT EXISTS (
            SELECT 1 FROM rnds r WHERE r.chave = f.chave
        )
        LIMIT 1000000
    ''')
    
    writer_gap = None
    batch_gaps = []
    for row in cursor:
        batch_gaps.append(row[0])
        
        if len(batch_gaps) >= BATCH_SIZE:
            table = pa.table({'CHAVE': batch_gaps})
            if writer_gap is None:
                writer_gap = pq.ParquetWriter(
                    f"{OUTPUT_DIR}/gap_faturamento_sem_rnds.parquet",
                    table.schema,
                    compression='zstd'
                )
            writer_gap.write_table(table)
            batch_gaps.clear()
    
    if batch_gaps:
        table = pa.table({'CHAVE': batch_gaps})
        if writer_gap is None:
            writer_gap = pq.ParquetWriter(
                f"{OUTPUT_DIR}/gap_faturamento_sem_rnds.parquet",
                table.schema,
                compression='zstd'
            )
        writer_gap.write_table(table)
    
    if writer_gap:
        writer_gap.close()
    
    print(f"    ✅ gap_faturamento_sem_rnds.parquet (amostra 1M)")
    
    # RNDS não faturado
    print("    Exportando RNDS não faturado...")
    cursor.execute('''
        SELECT r.chave
        FROM rnds r
        WHERE NOT EXISTS (
            SELECT 1 FROM faturamento f WHERE f.chave = r.chave
        )
        LIMIT 1000000
    ''')
    
    writer_rnds_gap = None
    batch_rnds_gaps = []
    for row in cursor:
        batch_rnds_gaps.append(row[0])
        
        if len(batch_rnds_gaps) >= BATCH_SIZE:
            table = pa.table({'CHAVE': batch_rnds_gaps})
            if writer_rnds_gap is None:
                writer_rnds_gap = pq.ParquetWriter(
                    f"{OUTPUT_DIR}/gap_rnds_nao_faturado.parquet",
                    table.schema,
                    compression='zstd'
                )
            writer_rnds_gap.write_table(table)
            batch_rnds_gaps.clear()
    
    if batch_rnds_gaps:
        table = pa.table({'CHAVE': batch_rnds_gaps})
        if writer_rnds_gap is None:
            writer_rnds_gap = pq.ParquetWriter(
                f"{OUTPUT_DIR}/gap_rnds_nao_faturado.parquet",
                table.schema,
                compression='zstd'
            )
        writer_rnds_gap.write_table(table)
    
    if writer_rnds_gap:
        writer_rnds_gap.close()
    
    print(f"    ✅ gap_rnds_nao_faturado.parquet (amostra 1M)")
    
    # Fechar conexão
    conn.close()
    
    # Remover DB temporário
    print("\n  Limpando DB temporário...")
    os.remove(db_path)
    
    return {
        'total_rnds': total_rnds,
        'rnds_unicas': rnds_unicas,
        'total_faturamento': total_faturamento,
        'matches': matches,
        'perc_match': perc_match,
        'gaps_faturamento': gaps_faturamento,
        'perc_faturamento_sem_rnds': perc_faturamento_sem_rnds,
        'rnds_nao_faturado': rnds_nao_faturado
    }

# ==================== EXECUÇÃO ====================

inicio_geral = datetime.now()

print("\n" + "="*80)
print("FASE 1: EXTRAÇÃO DE CHAVES")
print("="*80)

# 1. RNDS
arquivo_rnds = f"{OUTPUT_DIR}/chaves_rnds.parquet"
total_rnds = extrair_chaves_direto(
    r"base/RNDS_status_agendados.parquet",
    arquivo_rnds,
    extrair_mes_ano_rnds,
    'DATA_EXECUCAO'
)

# 2. SIA
arquivo_sia = f"{OUTPUT_DIR}/chaves_sia.parquet"
total_sia = extrair_chaves_direto(
    r"base/SIA_LIMPO.parquet",
    arquivo_sia,
    extrair_mes_ano_faturamento,
    'DT_CMP_FORMATADA'
)

# 3. SIH
arquivo_sih = f"{OUTPUT_DIR}/chaves_sih.parquet"
total_sih = extrair_chaves_direto(
    r"base/SIH_LIMPO.parquet",
    arquivo_sih,
    extrair_mes_ano_faturamento,
    'DT_CMP_FORMATADA'
)

# 4. UNIR SIA + SIH
print("\n" + "="*80)
print("FASE 2: UNIÃO SIA + SIH")
print("="*80)

arquivo_faturamento = f"{OUTPUT_DIR}/chaves_faturamento.parquet"
writer = None
total_faturamento = 0

try:
    print("  Copiando SIA...")
    pq_sia = pq.ParquetFile(arquivo_sia)
    for batch in pq_sia.iter_batches(batch_size=BATCH_SIZE):
        if writer is None:
            writer = pq.ParquetWriter(arquivo_faturamento, batch.schema, compression='zstd')
        writer.write_table(pa.Table.from_batches([batch]))
        total_faturamento += batch.num_rows
    
    print("  Adicionando SIH...")
    pq_sih = pq.ParquetFile(arquivo_sih)
    for batch in pq_sih.iter_batches(batch_size=BATCH_SIZE):
        writer.write_table(pa.Table.from_batches([batch]))
        total_faturamento += batch.num_rows
finally:
    if writer:
        writer.close()

print(f"✅ Faturamento: {total_faturamento:,} registros")

# 5. LINKAGE DISK-BASED
print("\n" + "="*80)
print("FASE 3: LINKAGE (DISK-BASED)")
print("="*80)

resultado = linkage_sqlite(arquivo_rnds, arquivo_faturamento)

# ==================== RESULTADOS ====================

print("\n" + "="*80)
print("📈 RESULTADOS FINAIS")
print("="*80)

print(f"\n📊 VOLUME:")
print(f"   • RNDS (total):        {resultado['total_rnds']:>15,}")
print(f"   • RNDS (únicas):       {resultado['rnds_unicas']:>15,}")
print(f"   • Faturamento:         {resultado['total_faturamento']:>15,}")

print(f"\n🔗 LINKAGE:")
print(f"   • Matches:             {resultado['matches']:>15,} ({resultado['perc_match']:.2f}%)")

print(f"\n⚠️  GAPS:")
print(f"   • RNDS não faturado:   {resultado['rnds_nao_faturado']:>15,} ({100-resultado['perc_match']:.2f}%)")
print(f"   • Faturamento sem RNDS:{resultado['gaps_faturamento']:>15,} ({resultado['perc_faturamento_sem_rnds']:.2f}%)")

# Salvar resumo
with open(f"{OUTPUT_DIR}/resumo_final.txt", "w", encoding="utf-8") as f:
    f.write("="*80 + "\n")
    f.write("ANÁLISE DE RASTREABILIDADE - SUS\n")
    f.write("="*80 + "\n\n")
    f.write(f"Data: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}\n\n")
    
    f.write("VOLUME:\n")
    f.write(f"  RNDS (total):    {resultado['total_rnds']:,}\n")
    f.write(f"  RNDS (únicas):   {resultado['rnds_unicas']:,}\n")
    f.write(f"  Faturamento:     {resultado['total_faturamento']:,}\n\n")
    
    f.write("LINKAGE:\n")
    f.write(f"  Matches: {resultado['matches']:,} ({resultado['perc_match']:.2f}%)\n\n")
    
    f.write("GAPS:\n")
    f.write(f"  RNDS não faturado:     {resultado['rnds_nao_faturado']:,}\n")
    f.write(f"  Faturamento sem RNDS:  {resultado['gaps_faturamento']:,}\n\n")
    
    f.write("INTERPRETAÇÃO:\n")
    f.write(f"  • {resultado['perc_match']:.1f}% dos procedimentos RNDS foram faturados\n")
    f.write(f"  • {100-resultado['perc_faturamento_sem_rnds']:.1f}% do faturamento passou pela regulação\n\n")
    
    f.write("ARQUIVOS:\n")
    f.write("  1. matches_rnds_faturamento.parquet\n")
    f.write("  2. gap_rnds_nao_faturado.parquet (amostra 1M)\n")
    f.write("  3. gap_faturamento_sem_rnds.parquet (amostra 1M)\n")

print(f"\n✅ Resumo: {OUTPUT_DIR}/resumo_final.txt")

# Tempo total
tempo_total = datetime.now() - inicio_geral
horas, resto = divmod(tempo_total.total_seconds(), 3600)
minutos, segundos = divmod(resto, 60)

print("\n" + "="*80)
print(f"⏱️  TEMPO TOTAL: {int(horas)}h {int(minutos)}min {int(segundos)}s")
print("="*80)
print("✅ ANÁLISE CONCLUÍDA!")
print("="*80)

🔵 ANÁLISE DE RASTREABILIDADE - DISK-BASED
Período: Jul/2024 a Jun/2025
Modo: Usa disco (SQLite) em vez de memória RAM

FASE 1: EXTRAÇÃO DE CHAVES

📊 RNDS_status_agendados.parquet
  5,000,000 proc | 4,975,096 válidas
  10,000,000 proc | 9,965,911 válidas
  15,000,000 proc | 14,836,471 válidas
  20,000,000 proc | 19,726,680 válidas
  25,000,000 proc | 24,686,545 válidas
  30,000,000 proc | 29,683,443 válidas
  35,000,000 proc | 34,669,242 válidas
  40,000,000 proc | 39,492,833 válidas
  45,000,000 proc | 44,427,139 válidas
  50,000,000 proc | 49,394,027 válidas
  55,000,000 proc | 54,389,762 válidas
  60,000,000 proc | 59,361,823 válidas
  65,000,000 proc | 64,179,329 válidas
  70,000,000 proc | 69,129,334 válidas
  75,000,000 proc | 74,101,285 válidas
  80,000,000 proc | 79,095,619 válidas
  85,000,000 proc | 84,049,506 válidas
  90,000,000 proc | 88,865,630 válidas
✅ 93,439,996 chaves | 0:06:07.756951

📊 SIA_LIMPO.parquet
  5,000,000 proc | 4,998,380 válidas
  10,000,000 proc | 9,993,1

In [32]:
# ==================== ANÁLISE COMPARATIVA: COM E SEM DATA ====================
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
import os
import sqlite3

BATCH_SIZE = 100_000
OUTPUT_DIR = "analise_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("🔵 ANÁLISE COMPARATIVA DE RASTREABILIDADE")
print("="*80)
print("Comparando 2 estratégias de chave:")
print("  1️⃣  CPF/CNS + SIGTAP (SEM data)")
print("  2️⃣  CPF/CNS + SIGTAP + MÊS/ANO (COM data)")
print("="*80)

# ==================== FUNÇÕES ====================

def extrair_mes_ano_rnds(data_str):
    if not data_str or str(data_str).strip() in ['', 'None', 'nan']:
        return None
    data_str = str(data_str).strip().split(' ')[0]
    try:
        if '/' in data_str:
            partes = data_str.split('/')
            if len(partes) == 3:
                return f"{partes[1].zfill(2)}{partes[2]}"
        elif '-' in data_str:
            partes = data_str.split('-')
            if len(partes) == 3:
                return f"{partes[1].zfill(2)}{partes[0]}"
    except:
        pass
    return None

def extrair_mes_ano_faturamento(data_str):
    if not data_str or str(data_str).strip() in ['', 'None', 'nan']:
        return None
    data_str = str(data_str).strip()
    try:
        if '/' in data_str:
            partes = data_str.split('/')
            if len(partes) == 2:
                return f"{partes[0].zfill(2)}{partes[1]}"
    except:
        pass
    return None

def criar_chaves_duplas(cpf, cns, sigtap, mes_ano):
    """Cria DUAS chaves: com e sem data"""
    identificador = cpf if cpf and cpf != '' else cns
    if identificador and sigtap:
        identificador = ''.join(filter(str.isdigit, str(identificador)))
        sigtap = ''.join(filter(str.isdigit, str(sigtap)))
        
        if len(identificador) in [11, 15] and len(sigtap) == 10:
            chave_sem_data = f"{identificador}|{sigtap}"
            chave_com_data = f"{identificador}|{sigtap}|{mes_ano}" if mes_ano else None
            return chave_sem_data, chave_com_data
    
    return None, None

def extrair_chaves_duplas(parquet_in, parquet_out_sem, parquet_out_com, extrair_data_fn, col_data):
    """Extrai DUAS versões de chaves: com e sem data"""
    print(f"\n📊 {os.path.basename(parquet_in)}")
    
    if not os.path.exists(parquet_in):
        print(f"❌ Não encontrado")
        return 0, 0
    
    total_proc = 0
    total_sem_data = 0
    total_com_data = 0
    
    writer_sem = None
    writer_com = None
    
    inicio = datetime.now()
    pq_file = pq.ParquetFile(parquet_in)
    
    try:
        for idx, batch in enumerate(pq_file.iter_batches(
            batch_size=BATCH_SIZE,
            columns=['CPF_PAC', 'CNS_PAC', 'COD_SIGTAP_PROCEDIMENTO', col_data]
        )):
            total_proc += batch.num_rows
            
            cpfs = batch['CPF_PAC'].to_pylist()
            cnss = batch['CNS_PAC'].to_pylist()
            sigtaps = batch['COD_SIGTAP_PROCEDIMENTO'].to_pylist()
            datas = batch[col_data].to_pylist()
            
            chaves_sem_data = []
            chaves_com_data = []
            
            for i in range(len(cpfs)):
                mes_ano = extrair_data_fn(datas[i])
                chave_sem, chave_com = criar_chaves_duplas(cpfs[i], cnss[i], sigtaps[i], mes_ano)
                
                if chave_sem:
                    chaves_sem_data.append(chave_sem)
                    total_sem_data += 1
                
                if chave_com:
                    chaves_com_data.append(chave_com)
                    total_com_data += 1
            
            # Salvar chaves SEM data
            if chaves_sem_data:
                table = pa.table({'CHAVE': chaves_sem_data})
                if writer_sem is None:
                    writer_sem = pq.ParquetWriter(parquet_out_sem, table.schema, compression='zstd')
                writer_sem.write_table(table)
                chaves_sem_data.clear()
            
            # Salvar chaves COM data
            if chaves_com_data:
                table = pa.table({'CHAVE': chaves_com_data})
                if writer_com is None:
                    writer_com = pq.ParquetWriter(parquet_out_com, table.schema, compression='zstd')
                writer_com.write_table(table)
                chaves_com_data.clear()
            
            if (idx + 1) % 50 == 0:
                print(f"  {total_proc:,} proc | Sem data: {total_sem_data:,} | Com data: {total_com_data:,}")
    
    finally:
        if writer_sem:
            writer_sem.close()
        if writer_com:
            writer_com.close()
    
    tempo = datetime.now() - inicio
    print(f"✅ Sem data: {total_sem_data:,} | Com data: {total_com_data:,} | {tempo}")
    
    return total_sem_data, total_com_data

def linkage_sqlite_simples(arquivo_rnds, arquivo_faturamento, tipo):
    """Linkage simples com SQLite"""
    print(f"\n🔗 LINKAGE {tipo}")
    
    db_path = f"{OUTPUT_DIR}/temp_{tipo}.db"
    
    if os.path.exists(db_path):
        os.remove(db_path)
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute('CREATE TABLE rnds (chave TEXT PRIMARY KEY)')
    cursor.execute('CREATE TABLE faturamento (chave TEXT)')
    cursor.execute('CREATE INDEX idx_fat ON faturamento(chave)')
    conn.commit()
    
    # Carregar RNDS
    print("  Carregando RNDS...")
    total_rnds = 0
    pq_rnds = pq.ParquetFile(arquivo_rnds)
    for batch in pq_rnds.iter_batches(batch_size=BATCH_SIZE, columns=['CHAVE']):
        chaves = batch['CHAVE'].to_pylist()
        cursor.executemany('INSERT OR IGNORE INTO rnds VALUES (?)', [(c,) for c in chaves])
        total_rnds += len(chaves)
        if total_rnds % 10_000_000 == 0:
            conn.commit()
            print(f"    RNDS: {total_rnds:,}")
    conn.commit()
    
    # Carregar Faturamento
    print("  Carregando Faturamento...")
    total_fat = 0
    pq_fat = pq.ParquetFile(arquivo_faturamento)
    for batch in pq_fat.iter_batches(batch_size=BATCH_SIZE, columns=['CHAVE']):
        chaves = batch['CHAVE'].to_pylist()
        cursor.executemany('INSERT INTO faturamento VALUES (?)', [(c,) for c in chaves])
        total_fat += len(chaves)
        if total_fat % 10_000_000 == 0:
            conn.commit()
            print(f"    Faturamento: {total_fat:,}")
    conn.commit()
    
    # Estatísticas
    print("  Calculando estatísticas...")
    
    cursor.execute('SELECT COUNT(*) FROM rnds')
    rnds_unicas = cursor.fetchone()[0]
    
    cursor.execute('''
        SELECT COUNT(DISTINCT f.chave)
        FROM faturamento f
        INNER JOIN rnds r ON f.chave = r.chave
    ''')
    matches = cursor.fetchone()[0]
    
    cursor.execute('''
        SELECT COUNT(*)
        FROM faturamento f
        WHERE NOT EXISTS (SELECT 1 FROM rnds r WHERE r.chave = f.chave)
    ''')
    gaps_fat = cursor.fetchone()[0]
    
    rnds_nao_fat = rnds_unicas - matches
    
    perc_match = (matches / total_rnds * 100) if total_rnds > 0 else 0
    perc_fat_sem_rnds = (gaps_fat / total_fat * 100) if total_fat > 0 else 0
    
    conn.close()
    os.remove(db_path)
    
    return {
        'total_rnds': total_rnds,
        'rnds_unicas': rnds_unicas,
        'total_faturamento': total_fat,
        'matches': matches,
        'perc_match': perc_match,
        'gaps_faturamento': gaps_fat,
        'perc_faturamento_sem_rnds': perc_fat_sem_rnds,
        'rnds_nao_faturado': rnds_nao_fat
    }

# ==================== EXECUÇÃO ====================

inicio_geral = datetime.now()

print("\n" + "="*80)
print("FASE 1: EXTRAÇÃO DE CHAVES (DUPLA)")
print("="*80)

# 1. RNDS
print("\n1️⃣  RNDS")
rnds_sem, rnds_com = extrair_chaves_duplas(
    r"base/RNDS_status_agendados.parquet",
    f"{OUTPUT_DIR}/rnds_sem_data.parquet",
    f"{OUTPUT_DIR}/rnds_com_data.parquet",
    extrair_mes_ano_rnds,
    'DATA_EXECUCAO'
)

# 2. SIA
print("\n2️⃣  SIA")
sia_sem, sia_com = extrair_chaves_duplas(
    r"base/SIA_LIMPO.parquet",
    f"{OUTPUT_DIR}/sia_sem_data.parquet",
    f"{OUTPUT_DIR}/sia_com_data.parquet",
    extrair_mes_ano_faturamento,
    'DT_CMP_FORMATADA'
)

# 3. SIH
print("\n3️⃣  SIH")
sih_sem, sih_com = extrair_chaves_duplas(
    r"base/SIH_LIMPO.parquet",
    f"{OUTPUT_DIR}/sih_sem_data.parquet",
    f"{OUTPUT_DIR}/sih_com_data.parquet",
    extrair_mes_ano_faturamento,
    'DT_CMP_FORMATADA'
)

# 4. UNIR SIA + SIH (SEM DATA)
print("\n" + "="*80)
print("FASE 2: UNIÃO SIA + SIH")
print("="*80)

print("\n📦 União SEM DATA")
writer = None
for batch in pq.ParquetFile(f"{OUTPUT_DIR}/sia_sem_data.parquet").iter_batches(batch_size=BATCH_SIZE):
    if writer is None:
        writer = pq.ParquetWriter(f"{OUTPUT_DIR}/faturamento_sem_data.parquet", batch.schema, compression='zstd')
    writer.write_table(pa.Table.from_batches([batch]))
for batch in pq.ParquetFile(f"{OUTPUT_DIR}/sih_sem_data.parquet").iter_batches(batch_size=BATCH_SIZE):
    writer.write_table(pa.Table.from_batches([batch]))
if writer:
    writer.close()
print("✅ Faturamento SEM data criado")

print("\n📦 União COM DATA")
writer = None
for batch in pq.ParquetFile(f"{OUTPUT_DIR}/sia_com_data.parquet").iter_batches(batch_size=BATCH_SIZE):
    if writer is None:
        writer = pq.ParquetWriter(f"{OUTPUT_DIR}/faturamento_com_data.parquet", batch.schema, compression='zstd')
    writer.write_table(pa.Table.from_batches([batch]))
for batch in pq.ParquetFile(f"{OUTPUT_DIR}/sih_com_data.parquet").iter_batches(batch_size=BATCH_SIZE):
    writer.write_table(pa.Table.from_batches([batch]))
if writer:
    writer.close()
print("✅ Faturamento COM data criado")

# 5. ANÁLISE COMPARATIVA
print("\n" + "="*80)
print("FASE 3: ANÁLISE COMPARATIVA")
print("="*80)

# Análise SEM DATA
print("\n" + "="*80)
print("📊 ANÁLISE 1: SEM DATA (CPF/CNS + SIGTAP)")
print("="*80)
resultado_sem_data = linkage_sqlite_simples(
    f"{OUTPUT_DIR}/rnds_sem_data.parquet",
    f"{OUTPUT_DIR}/faturamento_sem_data.parquet",
    "sem_data"
)

# Análise COM DATA
print("\n" + "="*80)
print("📊 ANÁLISE 2: COM DATA (CPF/CNS + SIGTAP + MÊS/ANO)")
print("="*80)
resultado_com_data = linkage_sqlite_simples(
    f"{OUTPUT_DIR}/rnds_com_data.parquet",
    f"{OUTPUT_DIR}/faturamento_com_data.parquet",
    "com_data"
)

# ==================== RESULTADOS COMPARATIVOS ====================

print("\n" + "="*80)
print("📊 RESULTADOS COMPARATIVOS")
print("="*80)

print("\n" + "="*80)
print("1️⃣  CHAVE SEM DATA (CPF/CNS + SIGTAP)")
print("="*80)
print(f"\n📊 VOLUME:")
print(f"   • RNDS (total):        {resultado_sem_data['total_rnds']:>15,}")
print(f"   • RNDS (únicas):       {resultado_sem_data['rnds_unicas']:>15,}")
print(f"   • Faturamento:         {resultado_sem_data['total_faturamento']:>15,}")

print(f"\n🔗 LINKAGE:")
print(f"   • Matches:             {resultado_sem_data['matches']:>15,} ({resultado_sem_data['perc_match']:.2f}%)")

print(f"\n⚠️  GAPS:")
print(f"   • RNDS não faturado:   {resultado_sem_data['rnds_nao_faturado']:>15,} ({100-resultado_sem_data['perc_match']:.2f}%)")
print(f"   • Faturamento sem RNDS:{resultado_sem_data['gaps_faturamento']:>15,} ({resultado_sem_data['perc_faturamento_sem_rnds']:.2f}%)")

print("\n" + "="*80)
print("2️⃣  CHAVE COM DATA (CPF/CNS + SIGTAP + MÊS/ANO)")
print("="*80)
print(f"\n📊 VOLUME:")
print(f"   • RNDS (total):        {resultado_com_data['total_rnds']:>15,}")
print(f"   • RNDS (únicas):       {resultado_com_data['rnds_unicas']:>15,}")
print(f"   • Faturamento:         {resultado_com_data['total_faturamento']:>15,}")

print(f"\n🔗 LINKAGE:")
print(f"   • Matches:             {resultado_com_data['matches']:>15,} ({resultado_com_data['perc_match']:.2f}%)")

print(f"\n⚠️  GAPS:")
print(f"   • RNDS não faturado:   {resultado_com_data['rnds_nao_faturado']:>15,} ({100-resultado_com_data['perc_match']:.2f}%)")
print(f"   • Faturamento sem RNDS:{resultado_com_data['gaps_faturamento']:>15,} ({resultado_com_data['perc_faturamento_sem_rnds']:.2f}%)")

print("\n" + "="*80)
print("📊 COMPARAÇÃO")
print("="*80)

diff_match = resultado_sem_data['perc_match'] - resultado_com_data['perc_match']
print(f"\n🎯 DIFERENÇA DE MATCH:")
print(f"   • SEM data: {resultado_sem_data['perc_match']:.2f}%")
print(f"   • COM data: {resultado_com_data['perc_match']:.2f}%")
print(f"   • Ganho sem data: +{diff_match:.2f} pontos percentuais")

print(f"\n💡 RECOMENDAÇÃO:")
if diff_match > 1:
    print(f"   ✅ Use chave SEM DATA (ganho de {diff_match:.1f}pp)")
    print(f"   Motivo: Datas podem ter formatos/erros diferentes")
else:
    print(f"   ⚠️  Diferença pequena ({diff_match:.1f}pp)")
    print(f"   Avaliar caso a caso")

# Salvar resumo comparativo
with open(f"{OUTPUT_DIR}/resumo_comparativo.txt", "w", encoding="utf-8") as f:
    f.write("="*80 + "\n")
    f.write("ANÁLISE COMPARATIVA DE RASTREABILIDADE - SUS\n")
    f.write("="*80 + "\n\n")
    f.write(f"Data: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}\n\n")
    
    f.write("="*80 + "\n")
    f.write("1. CHAVE SEM DATA (CPF/CNS + SIGTAP)\n")
    f.write("="*80 + "\n")
    f.write(f"RNDS (total):      {resultado_sem_data['total_rnds']:,}\n")
    f.write(f"RNDS (únicas):     {resultado_sem_data['rnds_unicas']:,}\n")
    f.write(f"Faturamento:       {resultado_sem_data['total_faturamento']:,}\n")
    f.write(f"Matches:           {resultado_sem_data['matches']:,} ({resultado_sem_data['perc_match']:.2f}%)\n")
    f.write(f"RNDS não faturado: {resultado_sem_data['rnds_nao_faturado']:,}\n\n")
    
    f.write("="*80 + "\n")
    f.write("2. CHAVE COM DATA (CPF/CNS + SIGTAP + MÊS/ANO)\n")
    f.write("="*80 + "\n")
    f.write(f"RNDS (total):      {resultado_com_data['total_rnds']:,}\n")
    f.write(f"RNDS (únicas):     {resultado_com_data['rnds_unicas']:,}\n")
    f.write(f"Faturamento:       {resultado_com_data['total_faturamento']:,}\n")
    f.write(f"Matches:           {resultado_com_data['matches']:,} ({resultado_com_data['perc_match']:.2f}%)\n")
    f.write(f"RNDS não faturado: {resultado_com_data['rnds_nao_faturado']:,}\n\n")
    
    f.write("="*80 + "\n")
    f.write("COMPARAÇÃO\n")
    f.write("="*80 + "\n")
    f.write(f"Diferença de match: +{diff_match:.2f} pp (sem data)\n\n")
    
    f.write("CONCLUSÃO:\n")
    if diff_match > 1:
        f.write(f"✅ Recomenda-se usar chave SEM DATA\n")
        f.write(f"   Ganho: {diff_match:.1f} pontos percentuais\n")
    else:
        f.write(f"⚠️  Diferença pequena entre as abordagens\n")

print(f"\n✅ Resumo: {OUTPUT_DIR}/resumo_comparativo.txt")

# Tempo total
tempo_total = datetime.now() - inicio_geral
horas, resto = divmod(tempo_total.total_seconds(), 3600)
minutos, segundos = divmod(resto, 60)

print("\n" + "="*80)
print(f"⏱️  TEMPO TOTAL: {int(horas)}h {int(minutos)}min {int(segundos)}s")
print("="*80)
print("✅ ANÁLISE COMPARATIVA CONCLUÍDA!")
print("="*80)

🔵 ANÁLISE COMPARATIVA DE RASTREABILIDADE
Comparando 2 estratégias de chave:
  1️⃣  CPF/CNS + SIGTAP (SEM data)
  2️⃣  CPF/CNS + SIGTAP + MÊS/ANO (COM data)

FASE 1: EXTRAÇÃO DE CHAVES (DUPLA)

1️⃣  RNDS

📊 RNDS_status_agendados.parquet
  5,000,000 proc | Sem data: 5,000,000 | Com data: 4,975,096
  10,000,000 proc | Sem data: 10,000,000 | Com data: 9,965,911
  15,000,000 proc | Sem data: 15,000,000 | Com data: 14,836,471
  20,000,000 proc | Sem data: 20,000,000 | Com data: 19,726,680
  25,000,000 proc | Sem data: 25,000,000 | Com data: 24,686,545
  30,000,000 proc | Sem data: 30,000,000 | Com data: 29,683,443
  35,000,000 proc | Sem data: 35,000,000 | Com data: 34,669,242
  40,000,000 proc | Sem data: 40,000,000 | Com data: 39,492,833
  45,000,000 proc | Sem data: 45,000,000 | Com data: 44,427,139
  50,000,000 proc | Sem data: 50,000,000 | Com data: 49,394,027
  55,000,000 proc | Sem data: 55,000,000 | Com data: 54,389,762
  60,000,000 proc | Sem data: 60,000,000 | Com data: 59,361,823


In [34]:
# ==================== DIAGNÓSTICO: POR QUE O MATCH ESTÁ BAIXO? ====================
import pyarrow.parquet as pq
import pandas as pd
from collections import Counter
import os

OUTPUT_DIR = "diagnostico"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("🔍 DIAGNÓSTICO: POR QUE O MATCH ESTÁ TÃO BAIXO?")
print("="*80)
print("Vamos investigar 5 hipóteses:")
print("  1️⃣  Identificadores diferentes (CPF vs CNS)")
print("  2️⃣  Formatos diferentes de SIGTAP")
print("  3️⃣  Amostra dos dados não batendo")
print("  4️⃣  Período de datas incompatível")
print("  5️⃣  Status da RNDS filtrados demais")
print("="*80)

BATCH_SIZE = 100_000

# ==================== DIAGNÓSTICO 1: IDENTIFICADORES ====================

print("\n" + "="*80)
print("1️⃣  DIAGNÓSTICO: IDENTIFICADORES (CPF vs CNS)")
print("="*80)

def analisar_identificadores(arquivo, nome, limit=10_000_000):
    """Analisa como os identificadores estão preenchidos"""
    print(f"\n📊 Analisando: {nome}")
    
    pq_file = pq.ParquetFile(arquivo)
    
    total = 0
    apenas_cpf = 0
    apenas_cns = 0
    ambos = 0
    nenhum = 0
    
    cpf_samples = []
    cns_samples = []
    
    for batch in pq_file.iter_batches(
        batch_size=BATCH_SIZE,
        columns=['CPF_PAC', 'CNS_PAC']
    ):
        if total >= limit:
            break
            
        df = batch.to_pandas()
        total += len(df)
        
        for _, row in df.iterrows():
            cpf = str(row['CPF_PAC']).strip() if pd.notna(row['CPF_PAC']) else ''
            cns = str(row['CNS_PAC']).strip() if pd.notna(row['CNS_PAC']) else ''
            
            tem_cpf = cpf and cpf != '' and cpf != 'None'
            tem_cns = cns and cns != '' and cns != 'None'
            
            if tem_cpf and tem_cns:
                ambos += 1
                if len(cpf_samples) < 5:
                    cpf_samples.append(cpf)
                if len(cns_samples) < 5:
                    cns_samples.append(cns)
            elif tem_cpf:
                apenas_cpf += 1
                if len(cpf_samples) < 5:
                    cpf_samples.append(cpf)
            elif tem_cns:
                apenas_cns += 1
                if len(cns_samples) < 5:
                    cns_samples.append(cns)
            else:
                nenhum += 1
        
        if total % 1_000_000 == 0:
            print(f"  Processados: {total:,}")
    
    print(f"\n  📊 Resultado (amostra de {total:,} registros):")
    print(f"     Apenas CPF:     {apenas_cpf:>10,} ({apenas_cpf/total*100:>5.1f}%)")
    print(f"     Apenas CNS:     {apenas_cns:>10,} ({apenas_cns/total*100:>5.1f}%)")
    print(f"     Ambos (CPF+CNS):{ambos:>10,} ({ambos/total*100:>5.1f}%)")
    print(f"     Nenhum:         {nenhum:>10,} ({nenhum/total*100:>5.1f}%)")
    
    if cpf_samples:
        print(f"\n  🔍 Amostras de CPF:")
        for cpf in cpf_samples[:3]:
            print(f"     {cpf}")
    
    if cns_samples:
        print(f"\n  🔍 Amostras de CNS:")
        for cns in cns_samples[:3]:
            print(f"     {cns}")
    
    return {
        'total': total,
        'apenas_cpf': apenas_cpf,
        'apenas_cns': apenas_cns,
        'ambos': ambos,
        'nenhum': nenhum
    }

# Analisar RNDS
rnds_ids = analisar_identificadores(
    r"base/RNDS_status_agendados.parquet",
    "RNDS",
    limit=5_000_000
)

# Analisar SIA
sia_ids = analisar_identificadores(
    r"base/SIA_LIMPO.parquet",
    "SIA",
    limit=5_000_000
)

# Analisar SIH
sih_ids = analisar_identificadores(
    r"base/SIH_LIMPO.parquet",
    "SIH",
    limit=1_000_000
)

# ==================== DIAGNÓSTICO 2: FORMATO SIGTAP ====================

print("\n" + "="*80)
print("2️⃣  DIAGNÓSTICO: FORMATO DOS CÓDIGOS SIGTAP")
print("="*80)

def analisar_sigtap(arquivo, nome, limit=1_000_000):
    """Analisa formatos dos códigos SIGTAP"""
    print(f"\n📊 Analisando: {nome}")
    
    pq_file = pq.ParquetFile(arquivo)
    
    total = 0
    formatos = Counter()
    amostras = []
    
    for batch in pq_file.iter_batches(
        batch_size=BATCH_SIZE,
        columns=['COD_SIGTAP_PROCEDIMENTO']
    ):
        if total >= limit:
            break
            
        df = batch.to_pandas()
        total += len(df)
        
        for _, row in df.iterrows():
            sigtap = str(row['COD_SIGTAP_PROCEDIMENTO']).strip()
            
            if sigtap and sigtap != 'None' and sigtap != '':
                # Verificar formato
                tem_ponto = '.' in sigtap
                tem_traco = '-' in sigtap
                tamanho = len(sigtap)
                
                formato = f"Tam:{tamanho}"
                if tem_ponto:
                    formato += "+PONTO"
                if tem_traco:
                    formato += "+TRAÇO"
                
                formatos[formato] += 1
                
                if len(amostras) < 10:
                    amostras.append(sigtap)
    
    print(f"\n  📊 Formatos encontrados (amostra de {total:,}):")
    for formato, qtd in formatos.most_common(5):
        print(f"     {formato:<20} {qtd:>10,} ({qtd/total*100:>5.1f}%)")
    
    print(f"\n  🔍 Amostras de códigos:")
    for codigo in amostras[:5]:
        print(f"     {codigo}")
    
    return formatos

# Analisar formatos
rnds_sigtap = analisar_sigtap(r"base/RNDS_status_agendados.parquet", "RNDS")
sia_sigtap = analisar_sigtap(r"base/SIA_LIMPO.parquet", "SIA")
sih_sigtap = analisar_sigtap(r"base/SIH_LIMPO.parquet", "SIH")

# ==================== DIAGNÓSTICO 3: COMPARAÇÃO DIRETA ====================

print("\n" + "="*80)
print("3️⃣  DIAGNÓSTICO: COMPARAÇÃO DIRETA DE AMOSTRAS")
print("="*80)

def extrair_amostra_chaves(arquivo, nome, limit=10_000):
    """Extrai uma amostra pequena de chaves"""
    print(f"\n📊 Extraindo amostra de: {nome}")
    
    pq_file = pq.ParquetFile(arquivo)
    chaves = set()
    
    for batch in pq_file.iter_batches(
        batch_size=BATCH_SIZE,
        columns=['CPF_PAC', 'CNS_PAC', 'COD_SIGTAP_PROCEDIMENTO']
    ):
        df = batch.to_pandas()
        
        for _, row in df.iterrows():
            if len(chaves) >= limit:
                break
                
            cpf = str(row['CPF_PAC']).strip() if pd.notna(row['CPF_PAC']) else ''
            cns = str(row['CNS_PAC']).strip() if pd.notna(row['CNS_PAC']) else ''
            sigtap = str(row['COD_SIGTAP_PROCEDIMENTO']).strip() if pd.notna(row['COD_SIGTAP_PROCEDIMENTO']) else ''
            
            # Limpar
            cpf = ''.join(filter(str.isdigit, cpf))
            cns = ''.join(filter(str.isdigit, cns))
            sigtap = ''.join(filter(str.isdigit, sigtap))
            
            identificador = cpf if cpf else cns
            
            if identificador and sigtap and len(sigtap) == 10:
                chave = f"{identificador}|{sigtap}"
                chaves.add(chave)
        
        if len(chaves) >= limit:
            break
    
    print(f"  ✅ Extraídas {len(chaves):,} chaves")
    return chaves

# Extrair amostras
print("\nExtraindo amostras...")
amostra_rnds = extrair_amostra_chaves(r"base/RNDS_status_agendados.parquet", "RNDS", 50_000)
amostra_sia = extrair_amostra_chaves(r"base/SIA_LIMPO.parquet", "SIA", 50_000)
amostra_sih = extrair_amostra_chaves(r"base/SIH_LIMPO.parquet", "SIH", 10_000)

amostra_faturamento = amostra_sia.union(amostra_sih)

# Calcular interseção
intersecao = amostra_rnds.intersection(amostra_faturamento)
perc = (len(intersecao) / len(amostra_rnds) * 100) if amostra_rnds else 0

print(f"\n  📊 Resultado da comparação de amostras:")
print(f"     RNDS:        {len(amostra_rnds):>10,} chaves")
print(f"     Faturamento: {len(amostra_faturamento):>10,} chaves")
print(f"     Interseção:  {len(intersecao):>10,} ({perc:.2f}%)")

if len(intersecao) > 0:
    print(f"\n  🎯 Exemplos de chaves que BATERAM:")
    for chave in list(intersecao)[:3]:
        print(f"     {chave}")

# ==================== DIAGNÓSTICO 4: ANÁLISE DE DATAS ====================

print("\n" + "="*80)
print("4️⃣  DIAGNÓSTICO: PERÍODO DAS DATAS")
print("="*80)

def analisar_periodo(arquivo, nome, col_data, limit=1_000_000):
    """Analisa o período das datas"""
    print(f"\n📊 Analisando: {nome}")
    
    pq_file = pq.ParquetFile(arquivo)
    
    meses = Counter()
    total = 0
    sem_data = 0
    
    for batch in pq_file.iter_batches(
        batch_size=BATCH_SIZE,
        columns=[col_data]
    ):
        if total >= limit:
            break
            
        df = batch.to_pandas()
        total += len(df)
        
        for _, row in df.iterrows():
            data = str(row[col_data]).strip()
            
            if data and data != 'None' and data != '':
                # Tentar extrair mês/ano
                if '/' in data:
                    partes = data.split('/')
                    if len(partes) >= 2:
                        mes = partes[-2] if len(partes) == 3 else partes[0]
                        ano = partes[-1]
                        mes_ano = f"{mes.zfill(2)}/{ano}"
                        meses[mes_ano] += 1
            else:
                sem_data += 1
    
    print(f"\n  📊 Período (amostra de {total:,}):")
    print(f"     Sem data: {sem_data:,} ({sem_data/total*100:.1f}%)")
    print(f"\n  📅 Meses mais frequentes:")
    for mes_ano, qtd in meses.most_common(10):
        print(f"     {mes_ano}: {qtd:>10,} ({qtd/total*100:>5.1f}%)")
    
    return meses

rnds_datas = analisar_periodo(r"base/RNDS_status_agendados.parquet", "RNDS", "DATA_EXECUCAO")
sia_datas = analisar_periodo(r"base/SIA_LIMPO.parquet", "SIA", "DT_CMP_FORMATADA")

# ==================== RESUMO E RECOMENDAÇÕES ====================

print("\n" + "="*80)
print("📋 RESUMO DO DIAGNÓSTICO")
print("="*80)

print("\n1️⃣  IDENTIFICADORES:")
print(f"   RNDS: {rnds_ids['apenas_cpf']/rnds_ids['total']*100:.1f}% só CPF | {rnds_ids['apenas_cns']/rnds_ids['total']*100:.1f}% só CNS")
print(f"   SIA:  {sia_ids['apenas_cpf']/sia_ids['total']*100:.1f}% só CPF | {sia_ids['apenas_cns']/sia_ids['total']*100:.1f}% só CNS")

print("\n2️⃣  FORMATO SIGTAP:")
print(f"   Verificar se formatos são compatíveis (veja acima)")

print("\n3️⃣  COMPARAÇÃO DIRETA:")
print(f"   Match em amostra: {perc:.2f}%")

print(f"\n💡 RECOMENDAÇÕES:")

if perc < 1:
    print(f"   ⚠️  Match MUITO baixo ({perc:.2f}%)")
    print(f"   Possíveis causas:")
    print(f"   • Identificadores incompatíveis (RNDS usa um, Faturamento usa outro)")
    print(f"   • Códigos SIGTAP em formatos diferentes")
    print(f"   • Dados da RNDS não correspondem ao faturamento")
    print(f"\n   🔧 Próximos passos:")
    print(f"   1. Comparar amostras individuais de CPF/CNS")
    print(f"   2. Verificar se há overlap de pacientes (só CPF ou só CNS)")
    print(f"   3. Testar match apenas por SIGTAP (sem identificador)")
else:
    print(f"   ✅ Match em amostra parece razoável ({perc:.1f}%)")

# Salvar relatório
with open(f"{OUTPUT_DIR}/relatorio_diagnostico.txt", "w", encoding="utf-8") as f:
    f.write("="*80 + "\n")
    f.write("RELATÓRIO DE DIAGNÓSTICO\n")
    f.write("="*80 + "\n\n")
    
    f.write("IDENTIFICADORES:\n")
    f.write(f"  RNDS:\n")
    f.write(f"    Apenas CPF: {rnds_ids['apenas_cpf']:,} ({rnds_ids['apenas_cpf']/rnds_ids['total']*100:.1f}%)\n")
    f.write(f"    Apenas CNS: {rnds_ids['apenas_cns']:,} ({rnds_ids['apenas_cns']/rnds_ids['total']*100:.1f}%)\n")
    f.write(f"    Ambos:      {rnds_ids['ambos']:,} ({rnds_ids['ambos']/rnds_ids['total']*100:.1f}%)\n\n")
    
    f.write(f"  SIA:\n")
    f.write(f"    Apenas CPF: {sia_ids['apenas_cpf']:,} ({sia_ids['apenas_cpf']/sia_ids['total']*100:.1f}%)\n")
    f.write(f"    Apenas CNS: {sia_ids['apenas_cns']:,} ({sia_ids['apenas_cns']/sia_ids['total']*100:.1f}%)\n")
    f.write(f"    Ambos:      {sia_ids['ambos']:,} ({sia_ids['ambos']/sia_ids['total']*100:.1f}%)\n\n")
    
    f.write("COMPARAÇÃO DE AMOSTRAS:\n")
    f.write(f"  RNDS:        {len(amostra_rnds):,}\n")
    f.write(f"  Faturamento: {len(amostra_faturamento):,}\n")
    f.write(f"  Match:       {len(intersecao):,} ({perc:.2f}%)\n\n")
    
    if perc < 1:
        f.write("CONCLUSÃO:\n")
        f.write("  ⚠️  Match muito baixo - investigar incompatibilidade de dados\n")

print(f"\n✅ Relatório salvo: {OUTPUT_DIR}/relatorio_diagnostico.txt")
print("\n" + "="*80)
print("✅ DIAGNÓSTICO CONCLUÍDO!")
print("="*80)

🔍 DIAGNÓSTICO: POR QUE O MATCH ESTÁ TÃO BAIXO?
Vamos investigar 5 hipóteses:
  1️⃣  Identificadores diferentes (CPF vs CNS)
  2️⃣  Formatos diferentes de SIGTAP
  3️⃣  Amostra dos dados não batendo
  4️⃣  Período de datas incompatível
  5️⃣  Status da RNDS filtrados demais

1️⃣  DIAGNÓSTICO: IDENTIFICADORES (CPF vs CNS)

📊 Analisando: RNDS
  Processados: 1,000,000
  Processados: 2,000,000
  Processados: 3,000,000
  Processados: 4,000,000
  Processados: 5,000,000

  📊 Resultado (amostra de 5,000,000 registros):
     Apenas CPF:              0 (  0.0%)
     Apenas CNS:         78,804 (  1.6%)
     Ambos (CPF+CNS): 4,921,196 ( 98.4%)
     Nenhum:                  0 (  0.0%)

  🔍 Amostras de CPF:
     78082722991
     02739653956
     10495251917

  🔍 Amostras de CNS:
     700006121205002
     705007625865458
     704000802859663

📊 Analisando: SIA
  Processados: 1,000,000
  Processados: 2,000,000
  Processados: 3,000,000
  Processados: 4,000,000
  Processados: 5,000,000

  📊 Resultado (am

In [36]:
# ==================== DIAGNÓSTICO COMPLETO + 3 ESTRATÉGIAS ====================
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
from collections import Counter
from datetime import datetime
import os
import sqlite3

OUTPUT_DIR = "diagnostico"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("🔍 DIAGNÓSTICO COMPLETO DE RASTREABILIDADE")
print("="*80)
print("Investigando 5 hipóteses:")
print("  1️⃣  Identificadores diferentes (CPF vs CNS)")
print("  2️⃣  Formatos diferentes de SIGTAP")
print("  3️⃣  Amostra dos dados não batendo")
print("  4️⃣  Período de datas incompatível")
print("  5️⃣  Status da RNDS filtrados demais")
print()
print("Comparando 3 estratégias de chave:")
print("  A) CPF/CNS (SEM SIGTAP e SEM data)")
print("  B) CPF/CNS + SIGTAP (SEM data)")
print("  C) CPF/CNS + SIGTAP + MÊS/ANO (COM data)")
print("="*80)

BATCH_SIZE = 100_000

# ==================== HIPÓTESE 1: IDENTIFICADORES ====================

print("\n" + "="*80)
print("HIPÓTESE 1: IDENTIFICADORES (CPF vs CNS)")
print("="*80)

def analisar_identificadores(arquivo, nome, limit=5_000_000):
    print(f"\n📊 {nome}")
    
    pq_file = pq.ParquetFile(arquivo)
    
    total = 0
    apenas_cpf = 0
    apenas_cns = 0
    ambos = 0
    nenhum = 0
    
    cpf_samples = []
    cns_samples = []
    
    for batch in pq_file.iter_batches(
        batch_size=BATCH_SIZE,
        columns=['CPF_PAC', 'CNS_PAC']
    ):
        if total >= limit:
            break
            
        df = batch.to_pandas()
        
        for _, row in df.iterrows():
            cpf = str(row['CPF_PAC']).strip() if pd.notna(row['CPF_PAC']) else ''
            cns = str(row['CNS_PAC']).strip() if pd.notna(row['CNS_PAC']) else ''
            
            tem_cpf = cpf and cpf != '' and cpf != 'None'
            tem_cns = cns and cns != '' and cns != 'None'
            
            if tem_cpf and tem_cns:
                ambos += 1
                if len(cpf_samples) < 3:
                    cpf_samples.append(cpf)
                if len(cns_samples) < 3:
                    cns_samples.append(cns)
            elif tem_cpf:
                apenas_cpf += 1
                if len(cpf_samples) < 3:
                    cpf_samples.append(cpf)
            elif tem_cns:
                apenas_cns += 1
                if len(cns_samples) < 3:
                    cns_samples.append(cns)
            else:
                nenhum += 1
        
        total = len(df) if total == 0 else total + len(df)
        
        if total >= limit:
            break
    
    print(f"  Amostra: {total:,} registros")
    print(f"  Apenas CPF:     {apenas_cpf:>10,} ({apenas_cpf/total*100:>5.1f}%)")
    print(f"  Apenas CNS:     {apenas_cns:>10,} ({apenas_cns/total*100:>5.1f}%)")
    print(f"  Ambos (CPF+CNS):{ambos:>10,} ({ambos/total*100:>5.1f}%)")
    print(f"  Nenhum:         {nenhum:>10,} ({nenhum/total*100:>5.1f}%)")
    
    if cpf_samples:
        print(f"  Amostras CPF: {', '.join(cpf_samples[:3])}")
    if cns_samples:
        print(f"  Amostras CNS: {', '.join(cns_samples[:3])}")
    
    return {
        'total': total,
        'apenas_cpf': apenas_cpf,
        'apenas_cns': apenas_cns,
        'ambos': ambos,
        'perc_cpf': apenas_cpf/total*100,
        'perc_cns': apenas_cns/total*100
    }

rnds_ids = analisar_identificadores(r"base/RNDS_status_agendados.parquet", "RNDS")
sia_ids = analisar_identificadores(r"base/SIA_LIMPO.parquet", "SIA")
sih_ids = analisar_identificadores(r"base/SIH_LIMPO.parquet", "SIH")

# ==================== HIPÓTESE 2: FORMATO SIGTAP ====================

print("\n" + "="*80)
print("HIPÓTESE 2: FORMATO DOS CÓDIGOS SIGTAP")
print("="*80)

def analisar_sigtap(arquivo, nome, limit=1_000_000):
    print(f"\n📊 {nome}")
    
    pq_file = pq.ParquetFile(arquivo)
    
    total = 0
    formatos = Counter()
    amostras = []
    
    for batch in pq_file.iter_batches(
        batch_size=BATCH_SIZE,
        columns=['COD_SIGTAP_PROCEDIMENTO']
    ):
        if total >= limit:
            break
            
        df = batch.to_pandas()
        
        for _, row in df.iterrows():
            sigtap = str(row['COD_SIGTAP_PROCEDIMENTO']).strip()
            
            if sigtap and sigtap != 'None' and sigtap != '':
                tem_ponto = '.' in sigtap
                tem_traco = '-' in sigtap
                tamanho = len(sigtap)
                
                formato = f"Tam{tamanho}"
                if tem_ponto:
                    formato += "+PONTO"
                if tem_traco:
                    formato += "+TRAÇO"
                
                formatos[formato] += 1
                
                if len(amostras) < 5:
                    amostras.append(sigtap)
        
        total += len(df)
        if total >= limit:
            break
    
    print(f"  Amostra: {total:,} registros")
    print(f"  Formatos encontrados:")
    for formato, qtd in formatos.most_common(3):
        print(f"    {formato:<20} {qtd:>10,} ({qtd/total*100:>5.1f}%)")
    print(f"  Amostras: {', '.join(amostras[:3])}")
    
    return formatos

rnds_sigtap = analisar_sigtap(r"base/RNDS_status_agendados.parquet", "RNDS")
sia_sigtap = analisar_sigtap(r"base/SIA_LIMPO.parquet", "SIA")
sih_sigtap = analisar_sigtap(r"base/SIH_LIMPO.parquet", "SIH")

# ==================== HIPÓTESE 3: COMPARAÇÃO DIRETA ====================

print("\n" + "="*80)
print("HIPÓTESE 3: AMOSTRA - COMPARAÇÃO DIRETA")
print("="*80)

def extrair_amostra(arquivo, nome, limit=50_000):
    print(f"\n📊 {nome}")
    
    pq_file = pq.ParquetFile(arquivo)
    chaves = set()
    
    for batch in pq_file.iter_batches(
        batch_size=BATCH_SIZE,
        columns=['CPF_PAC', 'CNS_PAC', 'COD_SIGTAP_PROCEDIMENTO']
    ):
        df = batch.to_pandas()
        
        for _, row in df.iterrows():
            if len(chaves) >= limit:
                break
                
            cpf = str(row['CPF_PAC']).strip() if pd.notna(row['CPF_PAC']) else ''
            cns = str(row['CNS_PAC']).strip() if pd.notna(row['CNS_PAC']) else ''
            sigtap = str(row['COD_SIGTAP_PROCEDIMENTO']).strip() if pd.notna(row['COD_SIGTAP_PROCEDIMENTO']) else ''
            
            cpf = ''.join(filter(str.isdigit, cpf))
            cns = ''.join(filter(str.isdigit, cns))
            sigtap = ''.join(filter(str.isdigit, sigtap))
            
            identificador = cpf if cpf else cns
            
            if identificador and sigtap and len(sigtap) == 10:
                chave = f"{identificador}|{sigtap}"
                chaves.add(chave)
        
        if len(chaves) >= limit:
            break
    
    print(f"  Extraídas: {len(chaves):,} chaves")
    return chaves

amostra_rnds = extrair_amostra(r"base/RNDS_status_agendados.parquet", "RNDS", 50_000)
amostra_sia = extrair_amostra(r"base/SIA_LIMPO.parquet", "SIA", 50_000)
amostra_sih = extrair_amostra(r"base/SIH_LIMPO.parquet", "SIH", 10_000)

amostra_faturamento = amostra_sia.union(amostra_sih)
intersecao = amostra_rnds.intersection(amostra_faturamento)
perc_amostra = (len(intersecao) / len(amostra_rnds) * 100) if amostra_rnds else 0

print(f"\n  📊 Resultado:")
print(f"    RNDS:        {len(amostra_rnds):>10,}")
print(f"    Faturamento: {len(amostra_faturamento):>10,}")
print(f"    Match:       {len(intersecao):>10,} ({perc_amostra:.2f}%)")

if len(intersecao) > 0:
    print(f"  Exemplos de match: {list(intersecao)[:2]}")

# ==================== HIPÓTESE 4: PERÍODO ====================

print("\n" + "="*80)
print("HIPÓTESE 4: PERÍODO DAS DATAS")
print("="*80)

def analisar_periodo(arquivo, nome, col_data, limit=1_000_000):
    print(f"\n📊 {nome}")
    
    pq_file = pq.ParquetFile(arquivo)
    
    meses = Counter()
    total = 0
    sem_data = 0
    
    for batch in pq_file.iter_batches(
        batch_size=BATCH_SIZE,
        columns=[col_data]
    ):
        if total >= limit:
            break
            
        df = batch.to_pandas()
        
        for _, row in df.iterrows():
            data = str(row[col_data]).strip()
            
            if data and data != 'None' and data != '':
                if '/' in data:
                    partes = data.split('/')
                    if len(partes) >= 2:
                        mes = partes[-2] if len(partes) == 3 else partes[0]
                        ano = partes[-1]
                        mes_ano = f"{mes.zfill(2)}/{ano}"
                        meses[mes_ano] += 1
            else:
                sem_data += 1
        
        total += len(df)
        if total >= limit:
            break
    
    print(f"  Amostra: {total:,} registros")
    print(f"  Sem data: {sem_data:,} ({sem_data/total*100:.1f}%)")
    print(f"  Top 5 meses:")
    for mes_ano, qtd in meses.most_common(5):
        print(f"    {mes_ano}: {qtd:>10,} ({qtd/total*100:>5.1f}%)")
    
    return meses

rnds_datas = analisar_periodo(r"base/RNDS_status_agendados.parquet", "RNDS", "DATA_EXECUCAO")
sia_datas = analisar_periodo(r"base/SIA_LIMPO.parquet", "SIA", "DT_CMP_FORMATADA")

# ==================== HIPÓTESE 5: STATUS RNDS ====================

print("\n" + "="*80)
print("HIPÓTESE 5: STATUS DA RNDS (FILTROS)")
print("="*80)

def analisar_status_completo(arquivo, limit=5_000_000):
    print(f"\n📊 Analisando todos os status da RNDS original")
    
    # Usar RNDS original (sem filtro de status)
    arquivo_original = r"base/RNDS.parquet"  # ou RNDS_renomeado.parquet
    
    if not os.path.exists(arquivo_original):
        print("  ⚠️  Arquivo RNDS original não encontrado, usando filtrado")
        arquivo_original = arquivo
    
    pq_file = pq.ParquetFile(arquivo_original)
    
    status_counter = Counter()
    total = 0
    
    for batch in pq_file.iter_batches(
        batch_size=BATCH_SIZE,
        columns=['st_solicitacao'] if 'st_solicitacao' in pq_file.schema.names else ['STATUS']
    ):
        if total >= limit:
            break
            
        df = batch.to_pandas()
        col_name = 'st_solicitacao' if 'st_solicitacao' in df.columns else 'STATUS'
        
        for _, row in df.iterrows():
            status = str(row[col_name]).strip() if pd.notna(row[col_name]) else 'Vazio'
            status_counter[status] += 1
        
        total += len(df)
        if total >= limit:
            break
    
    print(f"  Amostra: {total:,} registros")
    print(f"  Distribuição de status:")
    for status, qtd in status_counter.most_common(10):
        print(f"    {status:<35} {qtd:>10,} ({qtd/total*100:>5.1f}%)")
    
    return status_counter

status_completo = analisar_status_completo(r"base/RNDS_status_agendados.parquet")

# ==================== TESTE DAS 3 ESTRATÉGIAS ====================

print("\n" + "="*80)
print("TESTE DAS 3 ESTRATÉGIAS DE CHAVE")
print("="*80)

def criar_chaves_triplas(cpf, cns, sigtap, mes_ano):
    """Cria 3 versões de chaves"""
    identificador = cpf if cpf and cpf != '' else cns
    
    if not identificador:
        return None, None, None
    
    identificador = ''.join(filter(str.isdigit, str(identificador)))
    
    if len(identificador) not in [11, 15]:
        return None, None, None
    
    # Estratégia A: Apenas identificador
    chave_a = identificador
    
    # Estratégia B e C: Precisa de SIGTAP
    if sigtap:
        sigtap_limpo = ''.join(filter(str.isdigit, str(sigtap)))
        if len(sigtap_limpo) == 10:
            chave_b = f"{identificador}|{sigtap_limpo}"
            chave_c = f"{identificador}|{sigtap_limpo}|{mes_ano}" if mes_ano else None
            return chave_a, chave_b, chave_c
    
    return chave_a, None, None

def extrair_mes_ano_rnds(data_str):
    if not data_str or str(data_str).strip() in ['', 'None', 'nan']:
        return None
    data_str = str(data_str).strip().split(' ')[0]
    try:
        if '/' in data_str:
            partes = data_str.split('/')
            if len(partes) == 3:
                return f"{partes[1].zfill(2)}{partes[2]}"
        elif '-' in data_str:
            partes = data_str.split('-')
            if len(partes) == 3:
                return f"{partes[1].zfill(2)}{partes[0]}"
    except:
        pass
    return None

def extrair_mes_ano_faturamento(data_str):
    if not data_str or str(data_str).strip() in ['', 'None', 'nan']:
        return None
    data_str = str(data_str).strip()
    try:
        if '/' in data_str:
            partes = data_str.split('/')
            if len(partes) == 2:
                return f"{partes[0].zfill(2)}{partes[1]}"
    except:
        pass
    return None

print("\n🔄 Extraindo chaves (3 estratégias)...")

# Extrair amostras com as 3 estratégias
def extrair_triplo(arquivo, extrair_data_fn, col_data, limit=100_000):
    pq_file = pq.ParquetFile(arquivo)
    
    chaves_a = set()
    chaves_b = set()
    chaves_c = set()
    
    total = 0
    
    for batch in pq_file.iter_batches(
        batch_size=BATCH_SIZE,
        columns=['CPF_PAC', 'CNS_PAC', 'COD_SIGTAP_PROCEDIMENTO', col_data]
    ):
        if total >= limit:
            break
            
        df = batch.to_pandas()
        
        for _, row in df.iterrows():
            cpf = str(row['CPF_PAC']).strip() if pd.notna(row['CPF_PAC']) else ''
            cns = str(row['CNS_PAC']).strip() if pd.notna(row['CNS_PAC']) else ''
            sigtap = str(row['COD_SIGTAP_PROCEDIMENTO']).strip() if pd.notna(row['COD_SIGTAP_PROCEDIMENTO']) else ''
            data = row[col_data]
            
            mes_ano = extrair_data_fn(data)
            
            a, b, c = criar_chaves_triplas(cpf, cns, sigtap, mes_ano)
            
            if a:
                chaves_a.add(a)
            if b:
                chaves_b.add(b)
            if c:
                chaves_c.add(c)
        
        total += len(df)
        if total >= limit:
            break
    
    return chaves_a, chaves_b, chaves_c

print("  Extraindo RNDS...")
rnds_a, rnds_b, rnds_c = extrair_triplo(
    r"base/RNDS_status_agendados.parquet",
    extrair_mes_ano_rnds,
    'DATA_EXECUCAO',
    100_000
)

print("  Extraindo SIA...")
sia_a, sia_b, sia_c = extrair_triplo(
    r"base/SIA_LIMPO.parquet",
    extrair_mes_ano_faturamento,
    'DT_CMP_FORMATADA',
    100_000
)

print("  Extraindo SIH...")
sih_a, sih_b, sih_c = extrair_triplo(
    r"base/SIH_LIMPO.parquet",
    extrair_mes_ano_faturamento,
    'DT_CMP_FORMATADA',
    20_000
)

# Unir faturamento
fat_a = sia_a.union(sih_a)
fat_b = sia_b.union(sih_b)
fat_c = sia_c.union(sih_c)

# Calcular matches
match_a = rnds_a.intersection(fat_a)
match_b = rnds_b.intersection(fat_b)
match_c = rnds_c.intersection(fat_c)

perc_a = (len(match_a) / len(rnds_a) * 100) if rnds_a else 0
perc_b = (len(match_b) / len(rnds_b) * 100) if rnds_b else 0
perc_c = (len(match_c) / len(rnds_c) * 100) if rnds_c else 0

print("\n" + "="*80)
print("📊 RESULTADOS DAS 3 ESTRATÉGIAS")
print("="*80)

print(f"\n  A) CPF/CNS (SEM SIGTAP e SEM data)")
print(f"     RNDS:        {len(rnds_a):>10,}")
print(f"     Faturamento: {len(fat_a):>10,}")
print(f"     Match:       {len(match_a):>10,} ({perc_a:.2f}%)")

print(f"\n  B) CPF/CNS + SIGTAP (SEM data)")
print(f"     RNDS:        {len(rnds_b):>10,}")
print(f"     Faturamento: {len(fat_b):>10,}")
print(f"     Match:       {len(match_b):>10,} ({perc_b:.2f}%)")

print(f"\n  C) CPF/CNS + SIGTAP + MÊS/ANO (COM data)")
print(f"     RNDS:        {len(rnds_c):>10,}")
print(f"     Faturamento: {len(fat_c):>10,}")
print(f"     Match:       {len(match_c):>10,} ({perc_c:.2f}%)")

# ==================== CONCLUSÕES ====================

print("\n" + "="*80)
print("🎯 CONCLUSÕES E RECOMENDAÇÕES")
print("="*80)

print("\n1️⃣  IDENTIFICADORES:")
if abs(rnds_ids['perc_cpf'] - sia_ids['perc_cpf']) > 30:
    print(f"  ⚠️  INCOMPATIBILIDADE DETECTADA!")
    print(f"     RNDS: {rnds_ids['perc_cpf']:.0f}% CPF vs SIA: {sia_ids['perc_cpf']:.0f}% CPF")
    print(f"     Isso pode explicar o match baixo")
else:
    print(f"  ✅ Distribuição similar de identificadores")

print("\n2️⃣  FORMATOS SIGTAP:")
print(f"  Verificar formatos acima - podem estar diferentes")

print("\n3️⃣  MELHOR ESTRATÉGIA:")
melhor = max([('A', perc_a), ('B', perc_b), ('C', perc_c)], key=lambda x: x[1])
print(f"  🏆 Estratégia {melhor[0]}: {melhor[1]:.2f}% de match")

if perc_a > perc_b and perc_a > perc_c:
    print(f"  💡 Use apenas CPF/CNS (sem SIGTAP)")
    print(f"     Indica que o problema está no SIGTAP")
elif perc_b > perc_c:
    print(f"  💡 Use CPF/CNS + SIGTAP (sem data)")
    print(f"     Problema nas datas")
else:
    print(f"  💡 Match baixo em todas - investigar dados")

print("\n4️⃣  AÇÃO RECOMENDADA:")
if melhor[1] < 5:
    print(f"  ⚠️  Match muito baixo ({melhor[1]:.1f}%)")
    print(f"  Possíveis causas:")
    print(f"  • RNDS e Faturamento não correspondem aos mesmos pacientes")
    print(f"  • Problema sistêmico nos dados")
    print(f"  • Necessário revisar fonte dos dados")
else:
    print(f"  ✅ Use estratégia {melhor[0]} para análise final")
    print(f"     Match: {melhor[1]:.1f}%")

# Salvar relatório
with open(f"{OUTPUT_DIR}/relatorio_completo.txt", "w", encoding="utf-8") as f:
    f.write("="*80 + "\n")
    f.write("RELATÓRIO COMPLETO DE DIAGNÓSTICO\n")
    f.write("="*80 + "\n\n")
    
    f.write("IDENTIFICADORES:\n")
    f.write(f"  RNDS: {rnds_ids['perc_cpf']:.1f}% CPF | {rnds_ids['perc_cns']:.1f}% CNS\n")
    f.write(f"  SIA:  {sia_ids['perc_cpf']:.1f}% CPF | {sia_ids['perc_cns']:.1f}% CNS\n\n")
    
    f.write("ESTRATÉGIAS:\n")
    f.write(f"  A) CPF/CNS:             {perc_a:.2f}%\n")
    f.write(f"  B) CPF/CNS+SIGTAP:      {perc_b:.2f}%\n")
    f.write(f"  C) CPF/CNS+SIGTAP+DATA: {perc_c:.2f}%\n\n")
    
    f.write(f"RECOMENDAÇÃO: Usar estratégia {melhor[0]} ({melhor[1]:.2f}%)\n")

print(f"\n✅ Relatório: {OUTPUT_DIR}/relatorio_completo.txt")
print("\n" + "="*80)
print("✅ DIAGNÓSTICO COMPLETO CONCLUÍDO!")
print("="*80)

🔍 DIAGNÓSTICO COMPLETO DE RASTREABILIDADE
Investigando 5 hipóteses:
  1️⃣  Identificadores diferentes (CPF vs CNS)
  2️⃣  Formatos diferentes de SIGTAP
  3️⃣  Amostra dos dados não batendo
  4️⃣  Período de datas incompatível
  5️⃣  Status da RNDS filtrados demais

Comparando 3 estratégias de chave:
  A) CPF/CNS (SEM SIGTAP e SEM data)
  B) CPF/CNS + SIGTAP (SEM data)
  C) CPF/CNS + SIGTAP + MÊS/ANO (COM data)

HIPÓTESE 1: IDENTIFICADORES (CPF vs CNS)

📊 RNDS
  Amostra: 5,000,000 registros
  Apenas CPF:              0 (  0.0%)
  Apenas CNS:         78,804 (  1.6%)
  Ambos (CPF+CNS): 4,921,196 ( 98.4%)
  Nenhum:                  0 (  0.0%)
  Amostras CPF: 78082722991, 02739653956, 10495251917
  Amostras CNS: 700006121205002, 705007625865458, 704000802859663

📊 SIA
  Amostra: 5,000,000 registros
  Apenas CPF:         50,848 (  1.0%)
  Apenas CNS:      4,348,326 ( 87.0%)
  Ambos (CPF+CNS):   600,826 ( 12.0%)
  Nenhum:                  0 (  0.0%)
  Amostras CPF: 06949016875, 99320169420, 0

In [ ]:
# ==================== SOLUÇÃO: MATCH POR CPF E CNS SEPARADOS ====================
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
import os
import sqlite3

OUTPUT_DIR = "analise_final"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("🎯 SOLUÇÃO: MATCH POR CPF E CNS SEPARADAMENTE")
print("="*80)
print("Problema identificado:")
print("  • RNDS tem 98% com CPF+CNS")
print("  • SIA tem 87% APENAS CNS")
print("  • Priorizar CPF causa mismatch!")
print()
print("Solução:")
print("  ✅ Criar 2 chaves: uma por CPF, outra por CNS")
print("  ✅ Match se bater em QUALQUER uma")
print("="*80)

BATCH_SIZE = 100_000

# ==================== EXTRAÇÃO COM 2 CHAVES ====================

def criar_chaves_duplas_identificador(cpf, cns, sigtap):
    """Cria 2 chaves: uma por CPF, outra por CNS"""
    chaves = []
    
    # Limpar
    cpf_limpo = ''.join(filter(str.isdigit, str(cpf))) if cpf and str(cpf) != 'None' else ''
    cns_limpo = ''.join(filter(str.isdigit, str(cns))) if cns and str(cns) != 'None' else ''
    sigtap_limpo = ''.join(filter(str.isdigit, str(sigtap))) if sigtap and str(sigtap) != 'None' else ''
    
    # Validar SIGTAP
    if not sigtap_limpo or len(sigtap_limpo) != 10:
        return []
    
    # Chave por CPF
    if cpf_limpo and len(cpf_limpo) == 11:
        chaves.append(f"CPF:{cpf_limpo}|{sigtap_limpo}")
    
    # Chave por CNS
    if cns_limpo and len(cns_limpo) == 15:
        chaves.append(f"CNS:{cns_limpo}|{sigtap_limpo}")
    
    return chaves

def extrair_chaves_duplas_parquet(parquet_in, parquet_out):
    """Extrai chaves usando CPF E CNS"""
    print(f"\n📊 {os.path.basename(parquet_in)}")
    
    if not os.path.exists(parquet_in):
        print(f"❌ Não encontrado")
        return 0
    
    total_proc = 0
    total_val = 0
    writer = None
    
    inicio = datetime.now()
    pq_file = pq.ParquetFile(parquet_in)
    
    try:
        for idx, batch in enumerate(pq_file.iter_batches(
            batch_size=BATCH_SIZE,
            columns=['CPF_PAC', 'CNS_PAC', 'COD_SIGTAP_PROCEDIMENTO']
        )):
            total_proc += batch.num_rows
            
            cpfs = batch['CPF_PAC'].to_pylist()
            cnss = batch['CNS_PAC'].to_pylist()
            sigtaps = batch['COD_SIGTAP_PROCEDIMENTO'].to_pylist()
            
            chaves_batch = []
            
            for i in range(len(cpfs)):
                chaves = criar_chaves_duplas_identificador(cpfs[i], cnss[i], sigtaps[i])
                chaves_batch.extend(chaves)
                total_val += len(chaves)
            
            if chaves_batch:
                table = pa.table({'CHAVE': chaves_batch})
                if writer is None:
                    writer = pq.ParquetWriter(parquet_out, table.schema, compression='zstd')
                writer.write_table(table)
                chaves_batch.clear()
            
            if (idx + 1) % 50 == 0:
                print(f"  {total_proc:,} proc | {total_val:,} chaves")
    
    finally:
        if writer:
            writer.close()
    
    tempo = datetime.now() - inicio
    print(f"✅ {total_val:,} chaves | {tempo}")
    return total_val

# ==================== LINKAGE COM SQLite ====================

def linkage_duplo_sqlite(arquivo_rnds, arquivo_faturamento):
    """Linkage usando chaves duplas"""
    print(f"\n🔗 LINKAGE COM CHAVES DUPLAS")
    
    db_path = f"{OUTPUT_DIR}/temp_linkage.db"
    
    if os.path.exists(db_path):
        os.remove(db_path)
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute('CREATE TABLE rnds (chave TEXT PRIMARY KEY)')
    cursor.execute('CREATE TABLE faturamento (chave TEXT)')
    cursor.execute('CREATE INDEX idx_fat ON faturamento(chave)')
    conn.commit()
    
    # Carregar RNDS
    print("  Fase 1: Carregando RNDS...")
    total_rnds = 0
    pq_rnds = pq.ParquetFile(arquivo_rnds)
    for batch in pq_rnds.iter_batches(batch_size=BATCH_SIZE, columns=['CHAVE']):
        chaves = batch['CHAVE'].to_pylist()
        cursor.executemany('INSERT OR IGNORE INTO rnds VALUES (?)', [(c,) for c in chaves])
        total_rnds += len(chaves)
        if total_rnds % 10_000_000 == 0:
            conn.commit()
            print(f"    RNDS: {total_rnds:,}")
    conn.commit()
    print(f"  ✅ RNDS: {total_rnds:,}")
    
    # Carregar Faturamento
    print("\n  Fase 2: Carregando Faturamento...")
    total_fat = 0
    pq_fat = pq.ParquetFile(arquivo_faturamento)
    for batch in pq_fat.iter_batches(batch_size=BATCH_SIZE, columns=['CHAVE']):
        chaves = batch['CHAVE'].to_pylist()
        cursor.executemany('INSERT INTO faturamento VALUES (?)', [(c,) for c in chaves])
        total_fat += len(chaves)
        if total_fat % 10_000_000 == 0:
            conn.commit()
            print(f"    Faturamento: {total_fat:,}")
    conn.commit()
    print(f"  ✅ Faturamento: {total_fat:,}")
    
    # Estatísticas
    print("\n  Fase 3: Calculando estatísticas...")
    
    cursor.execute('SELECT COUNT(*) FROM rnds')
    rnds_unicas = cursor.fetchone()[0]
    print(f"    RNDS únicas: {rnds_unicas:,}")
    
    cursor.execute('''
        SELECT COUNT(DISTINCT f.chave)
        FROM faturamento f
        INNER JOIN rnds r ON f.chave = r.chave
    ''')
    matches = cursor.fetchone()[0]
    print(f"    Matches: {matches:,}")
    
    # Calcular por tipo
    cursor.execute('''
        SELECT COUNT(DISTINCT f.chave)
        FROM faturamento f
        INNER JOIN rnds r ON f.chave = r.chave
        WHERE f.chave LIKE 'CPF:%'
    ''')
    matches_cpf = cursor.fetchone()[0]
    
    cursor.execute('''
        SELECT COUNT(DISTINCT f.chave)
        FROM faturamento f
        INNER JOIN rnds r ON f.chave = r.chave
        WHERE f.chave LIKE 'CNS:%'
    ''')
    matches_cns = cursor.fetchone()[0]
    
    print(f"    Matches por CPF: {matches_cpf:,}")
    print(f"    Matches por CNS: {matches_cns:,}")
    
    # Percentuais
    perc_match = (matches / total_rnds * 100) if total_rnds > 0 else 0
    
    conn.close()
    os.remove(db_path)
    
    return {
        'total_rnds': total_rnds,
        'rnds_unicas': rnds_unicas,
        'total_faturamento': total_fat,
        'matches': matches,
        'matches_cpf': matches_cpf,
        'matches_cns': matches_cns,
        'perc_match': perc_match
    }

# ==================== EXECUÇÃO ====================

inicio_geral = datetime.now()

print("\n" + "="*80)
print("FASE 1: EXTRAÇÃO COM CHAVES DUPLAS (CPF E CNS)")
print("="*80)

# 1. RNDS
arquivo_rnds = f"{OUTPUT_DIR}/rnds_duplo.parquet"
total_rnds = extrair_chaves_duplas_parquet(
    r"base/RNDS_status_agendados.parquet",
    arquivo_rnds
)

# 2. SIA
arquivo_sia = f"{OUTPUT_DIR}/sia_duplo.parquet"
total_sia = extrair_chaves_duplas_parquet(
    r"base/SIA_LIMPO.parquet",
    arquivo_sia
)

# 3. SIH
arquivo_sih = f"{OUTPUT_DIR}/sih_duplo.parquet"
total_sih = extrair_chaves_duplas_parquet(
    r"base/SIH_LIMPO.parquet",
    arquivo_sih
)

# 4. UNIR SIA + SIH
print("\n" + "="*80)
print("FASE 2: UNIÃO SIA + SIH")
print("="*80)

arquivo_faturamento = f"{OUTPUT_DIR}/faturamento_duplo.parquet"
writer = None

try:
    print("  Copiando SIA...")
    pq_sia = pq.ParquetFile(arquivo_sia)
    for batch in pq_sia.iter_batches(batch_size=BATCH_SIZE):
        if writer is None:
            writer = pq.ParquetWriter(arquivo_faturamento, batch.schema, compression='zstd')
        writer.write_table(pa.Table.from_batches([batch]))
    
    print("  Adicionando SIH...")
    pq_sih = pq.ParquetFile(arquivo_sih)
    for batch in pq_sih.iter_batches(batch_size=BATCH_SIZE):
        writer.write_table(pa.Table.from_batches([batch]))
finally:
    if writer:
        writer.close()

print("✅ Faturamento unificado criado")

# 5. LINKAGE
print("\n" + "="*80)
print("FASE 3: ANÁLISE DE LINKAGE")
print("="*80)

resultado = linkage_duplo_sqlite(arquivo_rnds, arquivo_faturamento)

# ==================== RESULTADOS ====================

print("\n" + "="*80)
print("📊 RESULTADOS FINAIS")
print("="*80)

print(f"\n📊 VOLUME:")
print(f"   • RNDS (total chaves):     {resultado['total_rnds']:>15,}")
print(f"   • RNDS (únicas):           {resultado['rnds_unicas']:>15,}")
print(f"   • Faturamento:             {resultado['total_faturamento']:>15,}")

print(f"\n🔗 LINKAGE (CPF E CNS):")
print(f"   • Matches TOTAL:           {resultado['matches']:>15,} ({resultado['perc_match']:.2f}%)")
print(f"   • Matches por CPF:         {resultado['matches_cpf']:>15,}")
print(f"   • Matches por CNS:         {resultado['matches_cns']:>15,}")

print(f"\n💡 INTERPRETAÇÃO:")
if resultado['perc_match'] > 40:
    print(f"   🎉 EXCELENTE! {resultado['perc_match']:.1f}% de match")
    print(f"   ✅ RNDS é VÁLIDA para análise!")
    print(f"   A estratégia de chaves duplas funcionou!")
elif resultado['perc_match'] > 20:
    print(f"   ✅ BOM! {resultado['perc_match']:.1f}% de match")
    print(f"   RNDS pode ser usada com ressalvas")
elif resultado['perc_match'] > 5:
    print(f"   ⚠️  MODERADO: {resultado['perc_match']:.1f}% de match")
    print(f"   Investigar melhorias adicionais")
else:
    print(f"   ❌ BAIXO: {resultado['perc_match']:.1f}% de match")
    print(f"   Problema sistêmico nos dados")

# Salvar resumo
with open(f"{OUTPUT_DIR}/resumo_solucao_final.txt", "w", encoding="utf-8") as f:
    f.write("="*80 + "\n")
    f.write("SOLUÇÃO FINAL: MATCH POR CPF E CNS SEPARADOS\n")
    f.write("="*80 + "\n\n")
    f.write(f"Data: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}\n\n")
    
    f.write("ESTRATÉGIA:\n")
    f.write("  Criar 2 chaves para cada registro:\n")
    f.write("    1. CPF:{cpf}|{sigtap}\n")
    f.write("    2. CNS:{cns}|{sigtap}\n")
    f.write("  Match se bater em QUALQUER uma\n\n")
    
    f.write("VOLUME:\n")
    f.write(f"  RNDS (total):    {resultado['total_rnds']:,}\n")
    f.write(f"  RNDS (únicas):   {resultado['rnds_unicas']:,}\n")
    f.write(f"  Faturamento:     {resultado['total_faturamento']:,}\n\n")
    
    f.write("LINKAGE:\n")
    f.write(f"  Matches TOTAL:   {resultado['matches']:,} ({resultado['perc_match']:.2f}%)\n")
    f.write(f"  Matches por CPF: {resultado['matches_cpf']:,}\n")
    f.write(f"  Matches por CNS: {resultado['matches_cns']:,}\n\n")
    
    f.write("CONCLUSÃO:\n")
    if resultado['perc_match'] > 40:
        f.write("  ✅ RNDS é VÁLIDA para análise de rastreabilidade\n")
        f.write(f"  Match de {resultado['perc_match']:.1f}% é aceitável\n")
    elif resultado['perc_match'] > 20:
        f.write("  ⚠️  RNDS pode ser usada com ressalvas\n")
    else:
        f.write("  ❌ Match muito baixo - investigar dados\n")

print(f"\n✅ Resumo: {OUTPUT_DIR}/resumo_solucao_final.txt")

# Tempo total
tempo_total = datetime.now() - inicio_geral
horas, resto = divmod(tempo_total.total_seconds(), 3600)
minutos, segundos = divmod(resto, 60)

print("\n" + "="*80)
print(f"⏱️  TEMPO TOTAL: {int(horas)}h {int(minutos)}min {int(segundos)}s")
print("="*80)
print("✅ ANÁLISE CONCLUÍDA!")
print("="*80)
print("\n🎯 PRÓXIMOS PASSOS:")
print("   Se match > 40%: Use essa estratégia na análise final")
print("   Se match < 40%: Considere testar apenas por CNS")
print("="*80)

🎯 SOLUÇÃO: MATCH POR CPF E CNS SEPARADAMENTE
Problema identificado:
  • RNDS tem 98% com CPF+CNS
  • SIA tem 87% APENAS CNS
  • Priorizar CPF causa mismatch!

Solução:
  ✅ Criar 2 chaves: uma por CPF, outra por CNS
  ✅ Match se bater em QUALQUER uma

FASE 1: EXTRAÇÃO COM CHAVES DUPLAS (CPF E CNS)

📊 RNDS_status_agendados.parquet
  5,000,000 proc | 9,921,196 chaves
  10,000,000 proc | 19,852,890 chaves
  15,000,000 proc | 29,783,100 chaves
  20,000,000 proc | 39,714,257 chaves
  25,000,000 proc | 49,649,182 chaves
  30,000,000 proc | 59,571,534 chaves
  35,000,000 proc | 69,507,555 chaves
  40,000,000 proc | 79,437,726 chaves
  45,000,000 proc | 89,370,720 chaves
  50,000,000 proc | 99,305,293 chaves
  55,000,000 proc | 109,229,652 chaves
  60,000,000 proc | 119,163,421 chaves
  65,000,000 proc | 129,092,520 chaves
  70,000,000 proc | 139,029,405 chaves
  75,000,000 proc | 148,954,509 chaves
  80,000,000 proc | 158,878,665 chaves
  85,000,000 proc | 168,819,813 chaves
  90,000,000 proc 

In [ ]:
# ==================== ANÁLISE INVERTIDA + ENRIQUECIMENTO ====================
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
import os
import sqlite3

OUTPUT_DIR = "analise_rnds_focus"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("🎯 ANÁLISE FOCADA NA RNDS (INVERTIDA + ENRIQUECIDA)")
print("="*80)
print("Estratégia:")
print("  1️⃣  Criar mapa CNS→CPF da RNDS")
print("  2️⃣  Enriquecer Faturamento com CPFs da RNDS")
print("  3️⃣  Comparar: Quantos da RNDS foram FATURADOS?")
print()
print("Testando 3 estratégias de chave:")
print("  A) Apenas CPF/CNS (sem SIGTAP e sem data)")
print("  B) CPF/CNS + SIGTAP (sem data)")
print("  C) CPF/CNS + SIGTAP + MÊS/ANO (com data)")
print("="*80)

BATCH_SIZE = 100_000

# ==================== FUNÇÕES AUXILIARES ====================

def extrair_mes_ano_rnds(data_str):
    if not data_str or str(data_str).strip() in ['', 'None', 'nan']:
        return None
    data_str = str(data_str).strip().split(' ')[0]
    try:
        if '/' in data_str:
            partes = data_str.split('/')
            if len(partes) == 3:
                return f"{partes[1].zfill(2)}{partes[2]}"
        elif '-' in data_str:
            partes = data_str.split('-')
            if len(partes) == 3:
                return f"{partes[1].zfill(2)}{partes[0]}"
    except:
        pass
    return None

def extrair_mes_ano_faturamento(data_str):
    if not data_str or str(data_str).strip() in ['', 'None', 'nan']:
        return None
    data_str = str(data_str).strip()
    try:
        if '/' in data_str:
            partes = data_str.split('/')
            if len(partes) == 2:
                return f"{partes[0].zfill(2)}{partes[1]}"
    except:
        pass
    return None

def limpar_identificador(valor):
    if not valor or str(valor).strip() in ['', 'None', 'nan']:
        return ''
    return ''.join(filter(str.isdigit, str(valor)))

# ==================== PASSO 1: CRIAR MAPA CNS→CPF DA RNDS ====================

print("\n" + "="*80)
print("PASSO 1: CRIAR MAPA CNS→CPF DA RNDS")
print("="*80)

def criar_mapa_cns_cpf_rnds(arquivo_rnds, arquivo_mapa):
    """Cria mapa CNS→CPF da RNDS"""
    print(f"\n📊 Extraindo mapa CNS→CPF da RNDS...")
    
    pq_file = pq.ParquetFile(arquivo_rnds)
    
    mapa = {}
    total = 0
    
    for batch in pq_file.iter_batches(
        batch_size=BATCH_SIZE,
        columns=['CPF_PAC', 'CNS_PAC']
    ):
        df = batch.to_pandas()
        
        for _, row in df.iterrows():
            cpf = limpar_identificador(row['CPF_PAC'])
            cns = limpar_identificador(row['CNS_PAC'])
            
            # Se tem CNS válido, mapear para CPF
            if cns and len(cns) == 15:
                if cpf and len(cpf) == 11:
                    mapa[cns] = cpf  # CNS → CPF
        
        total += len(df)
        
        if total % 10_000_000 == 0:
            print(f"  Processados: {total:,} | Mapa: {len(mapa):,}")
    
    print(f"✅ Total processado: {total:,}")
    print(f"✅ Mapa CNS→CPF: {len(mapa):,} entradas")
    
    # Salvar mapa em Parquet
    if mapa:
        table = pa.table({
            'CNS': list(mapa.keys()),
            'CPF': list(mapa.values())
        })
        pq.write_table(table, arquivo_mapa, compression='zstd')
        print(f"✅ Mapa salvo: {arquivo_mapa}")
    
    return mapa

arquivo_mapa = f"{OUTPUT_DIR}/mapa_cns_cpf_rnds.parquet"
mapa_cns_cpf = criar_mapa_cns_cpf_rnds(
    r"base/RNDS_status_agendados.parquet",
    arquivo_mapa
)

# ==================== PASSO 2: EXTRAIR CHAVES (3 ESTRATÉGIAS) ====================

print("\n" + "="*80)
print("PASSO 2: EXTRAIR CHAVES DAS 3 BASES (3 ESTRATÉGIAS)")
print("="*80)

def criar_chaves_triplas(cpf, cns, sigtap, mes_ano, mapa=None):
    """Cria 3 versões de chaves, tentando enriquecer com mapa"""
    
    cpf = limpar_identificador(cpf)
    cns = limpar_identificador(cns)
    sigtap = limpar_identificador(sigtap)
    
    # Se não tem CPF mas tem CNS, tentar buscar no mapa
    if not cpf and cns and mapa and cns in mapa:
        cpf = mapa[cns]
    
    # Priorizar CPF (agora que está enriquecido)
    identificador = cpf if cpf and len(cpf) == 11 else (cns if cns and len(cns) == 15 else None)
    
    if not identificador:
        return None, None, None
    
    # A) Apenas identificador
    chave_a = identificador
    
    # B) Identificador + SIGTAP
    chave_b = f"{identificador}|{sigtap}" if sigtap and len(sigtap) == 10 else None
    
    # C) Identificador + SIGTAP + Data
    chave_c = f"{identificador}|{sigtap}|{mes_ano}" if chave_b and mes_ano else None
    
    return chave_a, chave_b, chave_c

def extrair_chaves_triplas_parquet(arquivo_in, arquivo_a, arquivo_b, arquivo_c, extrair_data_fn, col_data, mapa=None):
    """Extrai 3 tipos de chaves"""
    print(f"\n📊 {os.path.basename(arquivo_in)}")
    
    if not os.path.exists(arquivo_in):
        print(f"❌ Não encontrado")
        return 0, 0, 0
    
    total_proc = 0
    total_a = 0
    total_b = 0
    total_c = 0
    
    writer_a = None
    writer_b = None
    writer_c = None
    
    inicio = datetime.now()
    pq_file = pq.ParquetFile(arquivo_in)
    
    try:
        for idx, batch in enumerate(pq_file.iter_batches(
            batch_size=BATCH_SIZE,
            columns=['CPF_PAC', 'CNS_PAC', 'COD_SIGTAP_PROCEDIMENTO', col_data]
        )):
            total_proc += batch.num_rows
            
            df = batch.to_pandas()
            
            chaves_a = []
            chaves_b = []
            chaves_c = []
            
            for _, row in df.iterrows():
                mes_ano = extrair_data_fn(row[col_data])
                
                a, b, c = criar_chaves_triplas(
                    row['CPF_PAC'],
                    row['CNS_PAC'],
                    row['COD_SIGTAP_PROCEDIMENTO'],
                    mes_ano,
                    mapa
                )
                
                if a:
                    chaves_a.append(a)
                    total_a += 1
                if b:
                    chaves_b.append(b)
                    total_b += 1
                if c:
                    chaves_c.append(c)
                    total_c += 1
            
            # Salvar estratégia A
            if chaves_a:
                table = pa.table({'CHAVE': chaves_a})
                if writer_a is None:
                    writer_a = pq.ParquetWriter(arquivo_a, table.schema, compression='zstd')
                writer_a.write_table(table)
            
            # Salvar estratégia B
            if chaves_b:
                table = pa.table({'CHAVE': chaves_b})
                if writer_b is None:
                    writer_b = pq.ParquetWriter(arquivo_b, table.schema, compression='zstd')
                writer_b.write_table(table)
            
            # Salvar estratégia C
            if chaves_c:
                table = pa.table({'CHAVE': chaves_c})
                if writer_c is None:
                    writer_c = pq.ParquetWriter(arquivo_c, table.schema, compression='zstd')
                writer_c.write_table(table)
            
            if (idx + 1) % 50 == 0:
                print(f"  {total_proc:,} proc | A:{total_a:,} B:{total_b:,} C:{total_c:,}")
    
    finally:
        if writer_a:
            writer_a.close()
        if writer_b:
            writer_b.close()
        if writer_c:
            writer_c.close()
    
    tempo = datetime.now() - inicio
    print(f"✅ A:{total_a:,} B:{total_b:,} C:{total_c:,} | {tempo}")
    
    return total_a, total_b, total_c

# Extrair RNDS (sem mapa, ela é a fonte)
print("\n1️⃣  RNDS (sem enriquecimento)")
rnds_a, rnds_b, rnds_c = extrair_chaves_triplas_parquet(
    r"base/RNDS_status_agendados.parquet",
    f"{OUTPUT_DIR}/rnds_a.parquet",
    f"{OUTPUT_DIR}/rnds_b.parquet",
    f"{OUTPUT_DIR}/rnds_c.parquet",
    extrair_mes_ano_rnds,
    'DATA_EXECUCAO'
)

# Extrair SIA (COM enriquecimento)
print("\n2️⃣  SIA (COM enriquecimento via mapa RNDS)")
sia_a, sia_b, sia_c = extrair_chaves_triplas_parquet(
    r"base/SIA_LIMPO.parquet",
    f"{OUTPUT_DIR}/sia_a.parquet",
    f"{OUTPUT_DIR}/sia_b.parquet",
    f"{OUTPUT_DIR}/sia_c.parquet",
    extrair_mes_ano_faturamento,
    'DT_CMP_FORMATADA',
    mapa_cns_cpf  # USANDO O MAPA!
)

# Extrair SIH (COM enriquecimento)
print("\n3️⃣  SIH (COM enriquecimento via mapa RNDS)")
sih_a, sih_b, sih_c = extrair_chaves_triplas_parquet(
    r"base/SIH_LIMPO.parquet",
    f"{OUTPUT_DIR}/sih_a.parquet",
    f"{OUTPUT_DIR}/sih_b.parquet",
    f"{OUTPUT_DIR}/sih_c.parquet",
    extrair_mes_ano_faturamento,
    'DT_CMP_FORMATADA',
    mapa_cns_cpf  # USANDO O MAPA!
)

# ==================== PASSO 3: UNIR FATURAMENTO ====================

print("\n" + "="*80)
print("PASSO 3: UNIR SIA + SIH")
print("="*80)

for estrategia in ['a', 'b', 'c']:
    print(f"\n  União estratégia {estrategia.upper()}...")
    
    arquivo_fat = f"{OUTPUT_DIR}/faturamento_{estrategia}.parquet"
    writer = None
    
    try:
        # Copiar SIA
        pq_sia = pq.ParquetFile(f"{OUTPUT_DIR}/sia_{estrategia}.parquet")
        for batch in pq_sia.iter_batches(batch_size=BATCH_SIZE):
            if writer is None:
                writer = pq.ParquetWriter(arquivo_fat, batch.schema, compression='zstd')
            writer.write_table(pa.Table.from_batches([batch]))
        
        # Adicionar SIH
        pq_sih = pq.ParquetFile(f"{OUTPUT_DIR}/sih_{estrategia}.parquet")
        for batch in pq_sih.iter_batches(batch_size=BATCH_SIZE):
            writer.write_table(pa.Table.from_batches([batch]))
    finally:
        if writer:
            writer.close()
    
    print(f"  ✅ {arquivo_fat}")

# ==================== PASSO 4: LINKAGE (3 ESTRATÉGIAS) ====================

print("\n" + "="*80)
print("PASSO 4: ANÁLISE DE LINKAGE (3 ESTRATÉGIAS)")
print("="*80)

def linkage_simples_sqlite(arquivo_rnds, arquivo_faturamento, nome):
    """Linkage focado em: % da RNDS que foi faturada"""
    print(f"\n🔗 Estratégia {nome}")
    
    db_path = f"{OUTPUT_DIR}/temp_{nome}.db"
    
    if os.path.exists(db_path):
        os.remove(db_path)
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute('CREATE TABLE rnds (chave TEXT PRIMARY KEY)')
    cursor.execute('CREATE TABLE faturamento (chave TEXT)')
    cursor.execute('CREATE INDEX idx_fat ON faturamento(chave)')
    conn.commit()
    
    # Carregar RNDS
    print("  Carregando RNDS...")
    total_rnds = 0
    pq_rnds = pq.ParquetFile(arquivo_rnds)
    for batch in pq_rnds.iter_batches(batch_size=BATCH_SIZE, columns=['CHAVE']):
        chaves = batch['CHAVE'].to_pylist()
        cursor.executemany('INSERT OR IGNORE INTO rnds VALUES (?)', [(c,) for c in chaves])
        total_rnds += len(chaves)
        if total_rnds % 10_000_000 == 0:
            conn.commit()
            print(f"    {total_rnds:,}")
    conn.commit()
    
    # Carregar Faturamento
    print("  Carregando Faturamento...")
    total_fat = 0
    pq_fat = pq.ParquetFile(arquivo_faturamento)
    for batch in pq_fat.iter_batches(batch_size=BATCH_SIZE, columns=['CHAVE']):
        chaves = batch['CHAVE'].to_pylist()
        cursor.executemany('INSERT INTO faturamento VALUES (?)', [(c,) for c in chaves])
        total_fat += len(chaves)
        if total_fat % 10_000_000 == 0:
            conn.commit()
            print(f"    {total_fat:,}")
    conn.commit()
    
    # Estatísticas
    print("  Calculando...")
    
    cursor.execute('SELECT COUNT(*) FROM rnds')
    rnds_unicas = cursor.fetchone()[0]
    
    # INVERTIDO: Quantas da RNDS aparecem no Faturamento?
    cursor.execute('''
        SELECT COUNT(*)
        FROM rnds r
        WHERE EXISTS (SELECT 1 FROM faturamento f WHERE f.chave = r.chave)
    ''')
    rnds_faturadas = cursor.fetchone()[0]
    
    perc = (rnds_faturadas / rnds_unicas * 100) if rnds_unicas > 0 else 0
    
    conn.close()
    os.remove(db_path)
    
    return {
        'rnds_total': total_rnds,
        'rnds_unicas': rnds_unicas,
        'faturamento_total': total_fat,
        'rnds_faturadas': rnds_faturadas,
        'percentual': perc
    }

# Testar 3 estratégias
resultado_a = linkage_simples_sqlite(
    f"{OUTPUT_DIR}/rnds_a.parquet",
    f"{OUTPUT_DIR}/faturamento_a.parquet",
    "A"
)

resultado_b = linkage_simples_sqlite(
    f"{OUTPUT_DIR}/rnds_b.parquet",
    f"{OUTPUT_DIR}/faturamento_b.parquet",
    "B"
)

resultado_c = linkage_simples_sqlite(
    f"{OUTPUT_DIR}/rnds_c.parquet",
    f"{OUTPUT_DIR}/faturamento_c.parquet",
    "C"
)

# ==================== RESULTADOS ====================

print("\n" + "="*80)
print("📊 RESULTADOS COMPARATIVOS")
print("="*80)

print(f"\n  A) CPF/CNS (sem SIGTAP e sem data)")
print(f"     RNDS únicas:     {resultado_a['rnds_unicas']:>12,}")
print(f"     RNDS faturadas:  {resultado_a['rnds_faturadas']:>12,} ({resultado_a['percentual']:.2f}%)")

print(f"\n  B) CPF/CNS + SIGTAP (sem data)")
print(f"     RNDS únicas:     {resultado_b['rnds_unicas']:>12,}")
print(f"     RNDS faturadas:  {resultado_b['rnds_faturadas']:>12,} ({resultado_b['percentual']:.2f}%)")

print(f"\n  C) CPF/CNS + SIGTAP + MÊS/ANO")
print(f"     RNDS únicas:     {resultado_c['rnds_unicas']:>12,}")
print(f"     RNDS faturadas:  {resultado_c['rnds_faturadas']:>12,} ({resultado_c['percentual']:.2f}%)")

print("\n" + "="*80)
print("🎯 CONCLUSÃO")
print("="*80)

melhor = max([('A', resultado_a['percentual']), 
               ('B', resultado_b['percentual']), 
               ('C', resultado_c['percentual'])], 
              key=lambda x: x[1])

print(f"\n  🏆 Melhor estratégia: {melhor[0]} ({melhor[1]:.2f}%)")

if melhor[1] > 60:
    print(f"\n  🎉 EXCELENTE! {melhor[1]:.1f}% da RNDS foi faturada")
    print(f"  ✅ RNDS é ALTAMENTE CONFIÁVEL!")
elif melhor[1] > 40:
    print(f"\n  ✅ MUITO BOM! {melhor[1]:.1f}% da RNDS foi faturada")
    print(f"  ✅ RNDS é CONFIÁVEL para análise")
elif melhor[1] > 20:
    print(f"\n  ⚠️  MODERADO: {melhor[1]:.1f}% da RNDS foi faturada")
    print(f"  RNDS pode ser usada com ressalvas")
else:
    print(f"\n  ❌ BAIXO: {melhor[1]:.1f}% da RNDS foi faturada")
    print(f"  Investigar problemas nos dados")

# Salvar resumo
with open(f"{OUTPUT_DIR}/resumo_final_rnds.txt", "w", encoding="utf-8") as f:
    f.write("="*80 + "\n")
    f.write("ANÁLISE FOCADA NA RNDS (INVERTIDA + ENRIQUECIDA)\n")
    f.write("="*80 + "\n\n")
    f.write(f"Data: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}\n\n")
    
    f.write("ESTRATÉGIA:\n")
    f.write("  1. Criar mapa CNS→CPF da RNDS\n")
    f.write("  2. Enriquecer Faturamento com CPFs da RNDS\n")
    f.write("  3. Comparar: % da RNDS que foi FATURADA\n\n")
    
    f.write("RESULTADOS:\n")
    f.write(f"  A) Apenas identificador:  {resultado_a['percentual']:.2f}%\n")
    f.write(f"  B) Identificador+SIGTAP:  {resultado_b['percentual']:.2f}%\n")
    f.write(f"  C) Completo (com data):   {resultado_c['percentual']:.2f}%\n\n")
    
    f.write(f"MELHOR ESTRATÉGIA: {melhor[0]} ({melhor[1]:.2f}%)\n\n")
    
    f.write("CONCLUSÃO:\n")
    if melhor[1] > 40:
        f.write("  ✅ RNDS é VÁLIDA e CONFIÁVEL para análise de rastreabilidade\n")
    else:
        f.write("  ⚠️  RNDS tem limitações, usar com cuidado\n")

print(f"\n✅ Resumo: {OUTPUT_DIR}/resumo_final_rnds.txt")

tempo_total = datetime.now() - inicio_geral
horas, resto = divmod(tempo_total.total_seconds(), 3600)
minutos, segundos = divmod(resto, 60)

print("\n" + "="*80)
print(f"⏱️  TEMPO TOTAL: {int(horas)}h {int(minutos)}min {int(segundos)}s")
print("="*80)
print("✅ ANÁLISE CONCLUÍDA!")
print("="*80)

In [ ]:
fim = datetime.now()
tempo_total = fim - inicio

horas, resto = divmod(tempo_total.total_seconds(), 3600)
minutos, segundos = divmod(resto, 60)

print(f"✅ Tempo total de execução: {int(horas)}h {int(minutos)}min {int(segundos)}s")

✅ Tempo total de execução: 11h 55min 44s
